# Objective Quest DATAQUEST 5.0 — V9 TabICLv2 Full-Target Ordinal Foundation Probe

**Target:** `life_satisfaction` ordinal 1–4  
**Primary metric:** Quadratic Weighted Kappa (QWK)  
**Runtime target:** Kaggle T4 x2  
**Design contract:** this notebook implements the approved V9 design without adding new model families or leaderboard-driven tuning.

### Scientific question
Does a correctly executed **TabICLv2 full-target regressor** provide genuinely stronger or less-correlated ranking signal than the frozen V2 tree ensemble?

### Safety contract
- Competition data only. No external respondent data or test labels.
- Respondent `id` is the evaluation entity; duplicate respondents are collapsed for V9.
- V2 is run first as a frozen historical control before TabICL installation.
- All supervised V9 preprocessing decisions are outer-fold local.
- TabICL failure never falls back to V2 submission.
- Stage-S falsification gates are preregistered below; full CV/test inference runs only if the screen survives.
- Submission files are created only for candidates that pass the V9 submission gate.


## 0. Imports, seeds, configuration, and preregistered gates

The V2 control retains its historical modeling budget. V9 adds only the approved TabICLv2 screening/full-CV machinery.


In [1]:
import os, gc, json, random, re, sys, time, shutil, hashlib, platform, subprocess, importlib
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import minimize
from scipy.stats import spearmanr

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import cohen_kappa_score, mean_squared_error, mean_absolute_error, roc_auc_score

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

os.environ.setdefault("PYTHONHASHSEED", "42")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:512")


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(42)
pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 250)

class CFG:
    COMPETITION_NAME = "Objective Quest DATAQUEST 5.0"
    VERSION = "v9_tabiclv2_fulltarget"

    TRAIN_FILE = "train_final.csv"
    TEST_FILE = "test_final.csv"
    DICT_FILE = "data_dictionary.csv"
    SAMPLE_FILE = "sample_submission.csv"
    TARGET = "life_satisfaction"
    ID_COL = "id"
    HOUSEHOLD_COL = "household_id"

    OUTPUT_DIR = Path("/kaggle/working/oqdq5_v9") if Path("/kaggle/working").exists() else Path("./oqdq5_v9")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Historical V2 references only. Actual V2 control is recomputed in this notebook.
    V2_PUBLIC_LB_REFERENCE = 0.48193
    V2_EVALUATOR_ALIGNED_AUDIT_REFERENCE = 0.48062
    V1_CROSSFIT_OOF = 0.479662

    # Exact historical V2 repeated grouped CV.
    N_FOLDS = 5
    REPEAT_SEEDS = [42, 2026]
    N_REPEATS = len(REPEAT_SEEDS)
    INNER_N_FOLDS = 4
    INNER_TOP_N = 135
    INNER_FS_ESTIMATORS = 1800
    INNER_FS_EARLY_STOPPING = 100
    LGB_N_ESTIMATORS = 5500
    XGB_N_ESTIMATORS = 5500
    CAT_ITERATIONS = 5500
    LEARNING_RATE = 0.025
    EARLY_STOPPING = 220
    VERBOSE = 400
    USE_GPU = True
    CAT_DEVICES = "0:1"
    RUN_LGB_CATEGORICAL = True
    RUN_XGB_CATEGORICAL = True
    RUN_MONOTONIC = True
    RUN_MULTICLASS = True
    RUN_ORDINAL = True
    MIN_BRANCH_BLEND_GAIN = 0.00025
    GREEDY_ALPHA_GRID = np.round(np.arange(0.05, 0.56, 0.05), 2)
    MAX_SELECTED_SIDECARS = 4
    REPAIR_MIN_ACCURACY = 0.995
    REPAIR_MIN_SUPPORT = 100
    USE_SCALING = False
    USE_OVERSAMPLING = False
    USE_GENERIC_IMPUTATION = False

    EXPECTED_TRAIN_ROWS = 95609
    EXPECTED_TEST_ROWS = 23902
    EXPECTED_UNIQUE_TEST_IDS = 23666

    # V9 TabICLv2 provenance.
    TABICL_VERSION = "2.2.0"
    TABICL_HF_REPO = "jingang/TabICL"
    TABICL_REG_CHECKPOINT = "tabicl-regressor-v2-20260212.ckpt"
    TABICL_REG_SHA256 = "0db9cb538f114e79026bf08f45f41ad8dd7ad2de2aaca9a5ca8cd3bd9748ae7a"

    # V9 inference budgets from the approved design.
    TABICL_RUNTIME_N_ESTIMATORS = 1
    TABICL_SCREEN_N_ESTIMATORS = 2
    TABICL_FULL_N_ESTIMATORS = 4
    TABICL_BATCH_SIZE = 1
    TABICL_KV_CACHE = False
    TABICL_OFFLOAD_MODE = "auto"
    TABICL_USE_AMP = "auto"
    TABICL_USE_FA3 = "auto"
    TABICL_MISSING_FLAG_TOPK = 16
    TABICL_RESOURCE_TOPK = 96
    TABICL_SCREEN_FOLDS = [0, 2]

    # Stage-S preregistered gates.
    SCREEN_MIN_MEAN_DELTA_AUC = 0.005
    SCREEN_MIN_MEAN_DELTA_QWK = 0.002
    SCREEN_MAX_RESIDUAL_CORR_DIVERSITY = 0.98
    SCREEN_DIVERSITY_MIN_MEAN_QWK_GAIN = -0.001
    SCREEN_MIN_PER_FOLD_DELTA_AUC = -0.005  # operational definition of "no catastrophic regression"

    # Full-CV promotion/submission/breakthrough gates.
    SCIENTIFIC_MIN_QWK_GAIN = 0.002
    SUBMIT_MIN_QWK_GAIN = 0.0015
    BREAKTHROUGH_MIN_QWK_GAIN = 0.005
    BREAKTHROUGH_MIN_AUC_GAIN = 0.010
    SCIENTIFIC_BOOTSTRAP_P_POSITIVE = 0.95
    SUBMIT_BOOTSTRAP_P_POSITIVE = 0.90
    SUBMIT_MIN_POSITIVE_FOLDS = 3
    SUBMIT_MIN_WORST_FOLD_GAIN = -0.004
    MAX_TEST_OOF_CLASS_TV = 0.05
    BOOTSTRAP_REPS = 1000
    BLEND_ALPHA_GRID = [0.00, 0.25, 0.50, 0.75, 1.00]

    # Anchor sanity ranges are deliberately broad because GPU V2 is not bit-deterministic.
    V2_SANITY_QWK_MIN = 0.475
    V2_SANITY_QWK_MAX = 0.487
    V2_SANITY_AUC_GT1_MIN = 0.760

print("V9 output:", CFG.OUTPUT_DIR)
print("LightGBM:", lgb.__version__, "XGBoost:", xgb.__version__)
print("Preregistered screen folds:", CFG.TABICL_SCREEN_FOLDS)


V9 output: /kaggle/working/oqdq5_v9
LightGBM: 4.6.0 XGBoost: 3.2.0
Preregistered screen folds: [0, 2]


## 1. Competition data discovery and hard integrity checks


In [2]:
def discover_data_dir():
    required = {CFG.TRAIN_FILE, CFG.TEST_FILE, CFG.DICT_FILE, CFG.SAMPLE_FILE}
    roots = [Path("/kaggle/input"), Path("."), Path("./data")]
    candidates = []
    for root in roots:
        if not root.exists():
            continue
        try:
            for p in root.rglob(CFG.TRAIN_FILE):
                folder = p.parent
                present = {x.name for x in folder.iterdir() if x.is_file()}
                if required.issubset(present):
                    candidates.append(folder)
        except PermissionError:
            pass
    if not candidates:
        raise FileNotFoundError(f"Attach a Kaggle dataset containing {sorted(required)}")
    candidates = sorted(set(candidates), key=lambda p: (0 if str(p).startswith('/kaggle/input') else 1, len(str(p))))
    return candidates[0]


def sha256_file(path, chunk=2**20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

DATA_DIR = discover_data_dir()
TRAIN_PATH = DATA_DIR / CFG.TRAIN_FILE
TEST_PATH = DATA_DIR / CFG.TEST_FILE
DICT_PATH = DATA_DIR / CFG.DICT_FILE
SAMPLE_PATH = DATA_DIR / CFG.SAMPLE_FILE

train = pd.read_csv(TRAIN_PATH, low_memory=False)
test = pd.read_csv(TEST_PATH, low_memory=False)
data_dictionary = pd.read_csv(DICT_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

assert CFG.TARGET in train.columns and CFG.TARGET not in test.columns
assert len(train) == CFG.EXPECTED_TRAIN_ROWS, len(train)
assert len(test) == CFG.EXPECTED_TEST_ROWS, len(test)
assert test[CFG.ID_COL].nunique() == CFG.EXPECTED_UNIQUE_TEST_IDS
assert sorted(train[CFG.TARGET].dropna().unique().tolist()) == [1, 2, 3, 4]
assert train[CFG.ID_COL].notna().all() and test[CFG.ID_COL].notna().all()

# Same respondent must have exactly one label and exact duplicate payloads.
assert (train.groupby(CFG.ID_COL)[CFG.TARGET].nunique(dropna=False) <= 1).all()
for _, g in train[train.duplicated(CFG.ID_COL, keep=False)].groupby(CFG.ID_COL):
    assert len(g.drop_duplicates()) == 1, "Non-exact duplicate train respondent found"
for _, g in test[test.duplicated(CFG.ID_COL, keep=False)].groupby(CFG.ID_COL):
    assert len(g.drop_duplicates()) == 1, "Non-exact duplicate test respondent found"

# Prevent accidental test-template misuse.
if CFG.ID_COL in sample_submission.columns:
    assert len(sample_submission) != CFG.EXPECTED_UNIQUE_TEST_IDS or sample_submission[CFG.ID_COL].is_unique, (
        "Sample submission unexpectedly matches evaluator row contract; re-audit before continuing"
    )

DATA_MANIFEST = {
    "data_dir": str(DATA_DIR),
    "train_shape": list(train.shape),
    "test_shape": list(test.shape),
    "train_unique_ids": int(train[CFG.ID_COL].nunique()),
    "test_unique_ids": int(test[CFG.ID_COL].nunique()),
    "train_duplicate_id_rows": int(train.duplicated(CFG.ID_COL, keep=False).sum()),
    "test_duplicate_id_rows": int(test.duplicated(CFG.ID_COL, keep=False).sum()),
    "train_sha256": sha256_file(TRAIN_PATH),
    "test_sha256": sha256_file(TEST_PATH),
    "dictionary_sha256": sha256_file(DICT_PATH),
    "sample_sha256": sha256_file(SAMPLE_PATH),
}
with open(CFG.OUTPUT_DIR / "data_manifest.json", "w") as f:
    json.dump(DATA_MANIFEST, f, indent=2)
print(json.dumps(DATA_MANIFEST, indent=2))


{
  "data_dir": "/kaggle/input/datasets/tianchristiani150173/oq-dq-5",
  "train_shape": [
    95609,
    179
  ],
  "test_shape": [
    23902,
    178
  ],
  "train_unique_ids": 94663,
  "test_unique_ids": 23666,
  "train_duplicate_id_rows": 1892,
  "test_duplicate_id_rows": 472,
  "train_sha256": "0ccc7f7aa3d9e7520d1edd0cae30c1ac119fcb4c508fdd7c9ebc871ceaee2056",
  "test_sha256": "c7027da38aba915e6a9952d21c6cd960b22c172e0ccf3f9f570d63d07e008740",
  "dictionary_sha256": "4ee1269e4e10fd62225916bb8eff1cd51930a211cb8488d6c97a5b4817e4f631",
  "sample_sha256": "3b27035c9910f24c7a0cf29f85c1382be423f58cda0524bbd7b0a24ab9e65abf"
}


## 2. Frozen V2 control

This section is copied from the historical V2 modeling path, except the EDA-only probe/plots and V2 submission export are omitted. It is executed **before** TabICL installation so the control cannot be changed by foundation-model dependency resolution.


In [3]:
y = train[CFG.TARGET].astype(int).values
groups = train[CFG.ID_COL].astype(str).values
survey_year_for_strata = train["survey_year"].fillna(-1).astype(str) if "survey_year" in train.columns else pd.Series("NA", index=train.index)
strata = train[CFG.TARGET].astype(str) + "__yr_" + survey_year_for_strata

repeated_folds = []
fold_audit_rows = []
for repeat, seed in enumerate(CFG.REPEAT_SEEDS):
    cv = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=seed)
    folds = list(cv.split(train, strata, groups=groups))
    for fold, (tr_idx, va_idx) in enumerate(folds):
        overlap = set(groups[tr_idx]).intersection(set(groups[va_idx]))
        assert not overlap
        repeated_folds.append((repeat, fold, seed, tr_idx, va_idx))
        fold_audit_rows.append({
            "repeat": repeat, "fold": fold, "seed": seed,
            "train_n": len(tr_idx), "valid_n": len(va_idx),
            "group_overlap": len(overlap),
            "valid_target_mean": float(y[va_idx].mean()),
        })
display(pd.DataFrame(fold_audit_rows))


,repeat,fold,seed,train_n,valid_n,group_overlap,valid_target_mean
0,0,0,42,76485,19124,0,1.606672
1,0,1,42,76500,19109,0,1.605893
2,0,2,42,76487,19122,0,1.611024
3,0,3,42,76491,19118,0,1.604090
4,0,4,42,76473,19136,0,1.604776
5,1,0,2026,76501,19108,0,1.604459
6,1,1,2026,76477,19132,0,1.605373
7,1,2,2026,76476,19133,0,1.610255
8,1,3,2026,76483,19126,0,1.611994
9,1,4,2026,76499,19110,0,1.600366


In [4]:
REPAIR_PAIRS = [
    ("data_year", "survey_year"),
    ("survey_year", "data_year"),
    ("interview_month", "interview_quarter"),
    ("food_security_4cat", "food_security_3cat"),
    ("cancer_types_count", "cancer_ever"),
    ("ecigarette_status", "ever_used_ecigarette"),
    ("children_in_family", "children_in_household"),
    ("children_in_household", "children_in_family"),
    ("smoking_status", "ever_smoked"),
    ("health_insurance_type", "private_health_insurance"),
    ("marital_status_detailed", "marital_status"),
]

SPECIAL_CODES = {
    "weight_pounds": {996, 997, 999},
    "height_inches": {96, 97, 98, 99},
    "age": {97, 99},
    "hours_worked_per_week": {990},
    "days_missed_work": {997, 998, 999},
    "income_to_poverty_ratio": {110},
}


def learn_mode_mapping(df, src, dst, min_support=100):
    if src not in df.columns or dst not in df.columns:
        return None
    pairs = df[[src, dst]].dropna()
    if len(pairs) < min_support:
        return None
    mapping = pairs.groupby(src)[dst].agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan).to_dict()
    pred = pairs[src].map(mapping)
    return {"mapping": mapping, "accuracy": float((pred.values == pairs[dst].values).mean()), "support": len(pairs)}


class FoldPreprocessor:
    def __init__(self, min_accuracy=0.995, min_support=100):
        self.min_accuracy = min_accuracy
        self.min_support = min_support
        self.repair_rules_ = {}
        self.repair_report_ = None

    def fit(self, df):
        rows=[]
        for src,dst in REPAIR_PAIRS:
            learned = learn_mode_mapping(df, src, dst, self.min_support)
            if learned is None:
                continue
            accepted = learned["accuracy"] >= self.min_accuracy
            rows.append({"source":src,"destination":dst,"accuracy":learned["accuracy"],"support":learned["support"],"accepted":accepted})
            if accepted:
                self.repair_rules_[(src,dst)] = learned["mapping"]
        self.repair_report_ = pd.DataFrame(rows)
        return self

    def transform(self, df):
        out = df.drop(columns=[CFG.TARGET], errors="ignore").copy()
        base_cols = [c for c in out.columns if c not in [CFG.ID_COL, CFG.HOUSEHOLD_COL]]
        out["row_missing_count_raw"] = out[base_cols].isna().sum(axis=1).astype(np.int16)

        before_missing = out[base_cols].isna().sum(axis=1)
        for (src,dst),mapping in self.repair_rules_.items():
            if src not in out or dst not in out:
                continue
            mask = out[dst].isna() & out[src].notna()
            if mask.any():
                inferred = out.loc[mask,src].map(mapping)
                idx = inferred[inferred.notna()].index
                out.loc[idx,dst] = inferred.loc[idx]
        after_missing = out[base_cols].isna().sum(axis=1)
        out["row_missing_count_after_repair"] = after_missing.astype(np.int16)
        out["repaired_value_count"] = (before_missing-after_missing).astype(np.int16)

        special_cols=[]
        for col,codes in SPECIAL_CODES.items():
            if col in out.columns:
                flag = f"{col}__special_code"
                out[flag] = out[col].isin(codes).astype(np.int8)
                special_cols.append(flag)
        out["special_code_count"] = out[special_cols].sum(axis=1).astype(np.int16) if special_cols else 0
        return out


In [5]:
def parse_valid_codes(value_text):
    if pd.isna(value_text):
        return None
    s = str(value_text).strip()
    if not s or s.lower() in {"numerik", "nilai tahun", "kode numerik"}:
        return None

    # Explicit mappings: 1=..., 2=..., ...
    matches = re.findall(r"(?:^|,)\s*(-?\d+(?:\.\d+)?)\s*=", s)
    if matches:
        vals = sorted({float(x) for x in matches})
        return vals if 1 <= len(vals) <= 30 else None

    # Simple integer range, e.g. 1-12 or 18-85. Keep only compact coded ranges.
    m = re.fullmatch(r"\s*(-?\d+)\s*-\s*(-?\d+)\+?\s*", s)
    if m:
        lo,hi = map(int,m.groups())
        if 0 <= hi-lo <= 30:
            return [float(x) for x in range(lo,hi+1)]
        return None

    # Bare comma-separated numeric list (e.g. survey years).
    parts = [p.strip() for p in s.split(',')]
    if 1 < len(parts) <= 30:
        try:
            return sorted({float(p) for p in parts})
        except Exception:
            return None
    return None


def build_dictionary_valid_code_map(dictionary_df, feature_names):
    result={}
    if not {"kolom","nilai"}.issubset(dictionary_df.columns):
        return result
    for _,row in dictionary_df.iterrows():
        col=str(row["kolom"])
        if col not in feature_names:
            continue
        vals=parse_valid_codes(row["nilai"])
        if vals is not None:
            result[col]=set(vals)
    return result

DICT_VALID_CODES = build_dictionary_valid_code_map(data_dictionary, set(test.columns))
print("Dictionary-coded columns parsed:", len(DICT_VALID_CODES))

# Only create semantic copies where nonstandard observed values actually exist.
combined_for_audit = pd.concat([train.drop(columns=[CFG.TARGET]), test], axis=0, ignore_index=True)
SEMANTIC_CODE_COLS=[]
semantic_audit=[]
for col,valid in DICT_VALID_CODES.items():
    if col not in combined_for_audit:
        continue
    numeric = pd.to_numeric(combined_for_audit[col], errors="coerce")
    nonstd = numeric.notna() & ~numeric.isin(valid)
    if nonstd.any():
        SEMANTIC_CODE_COLS.append(col)
        semantic_audit.append({
            "feature":col,
            "valid_codes":sorted(valid),
            "nonstandard_rows":int(nonstd.sum()),
            "nonstandard_rate":float(nonstd.mean()),
            "observed_nonstandard":sorted(numeric[nonstd].dropna().unique().tolist())[:20],
        })

semantic_audit_df = pd.DataFrame(semantic_audit).sort_values("nonstandard_rate", ascending=False)
print("Columns receiving dual semantic representation:", len(SEMANTIC_CODE_COLS))
display(semantic_audit_df.head(40))


Dictionary-coded columns parsed: 161
Columns receiving dual semantic representation: 150


,feature,valid_codes,nonstandard_rows,nonstandard_rate,observed_nonstandard
68,child_government_insurance,"[1.0, 2.0]",110110,0.921338,"[3.0, 7.0, 9.0]"
70,other_government_insurance,"[1.0, 2.0]",109750,0.918326,"[3.0, 7.0, 9.0]"
69,state_health_plan,"[1.0, 2.0]",109308,0.914627,"[3.0, 7.0, 9.0]"
71,military_health_insurance,"[1.0, 2.0]",103164,0.863218,"[3.0, 7.0, 9.0]"
52,low_income_government_insurance,"[1.0, 2.0]",96387,0.806512,"[3.0, 7.0, 9.0]"
141,income_poverty_category,"[1.0, 2.0, 3.0, 4.0]",94434,0.790170,"[5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13..."
123,legal_marital_status,"[1.0, 2.0]",91685,0.767168,"[3.0, 4.0, 5.0, 9.0]"
124,parental_status,"[1.0, 2.0]",82382,0.689326,"[3.0, 9.0]"
5,max_education_in_family,"[1.0, 2.0, 3.0, 4.0]",82222,0.687987,"[5.0, 6.0, 7.0, 8.0, 9.0, 10.0]"
3,education_level,"[1.0, 2.0, 3.0, 4.0]",73348,0.613734,"[5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 97.0, 99.0, 99..."


In [6]:
def qwk(y_true, y_pred_class):
    return cohen_kappa_score(np.asarray(y_true), np.asarray(y_pred_class), weights="quadratic")


def apply_thresholds(pred, thresholds):
    t=np.sort(np.asarray(thresholds,dtype=float))
    return np.digitize(np.asarray(pred), bins=t).astype(int)+1


def initial_thresholds_from_distribution(y_true, continuous_pred):
    counts=pd.Series(y_true).value_counts(normalize=True).sort_index()
    cum=np.cumsum([counts.get(k,0.0) for k in [1,2,3]])
    return np.quantile(continuous_pred, np.clip(cum,0.001,0.999))


def optimize_thresholds(y_true, continuous_pred, starts=None, maxiter=160):
    y_true=np.asarray(y_true)
    p=np.asarray(continuous_pred,dtype=float)
    if starts is None:
        starts=[initial_thresholds_from_distribution(y_true,p), np.array([1.55,2.03,2.52])]
    best=None
    for start in starts:
        start=np.sort(np.asarray(start,dtype=float))
        def objective(t):
            ts=np.sort(t)
            if np.min(np.diff(ts)) < 0.03:
                return 1e3
            return -qwk(y_true, apply_thresholds(p,ts))
        res=minimize(objective,start,method="Nelder-Mead",options={"maxiter":maxiter,"xatol":1e-5,"fatol":1e-7})
        ts=np.sort(res.x)
        score=qwk(y_true,apply_thresholds(p,ts))
        if best is None or score>best[1]:
            best=(ts,score)
    return best


In [7]:
FUNCTIONAL_DIFFICULTY_COLS = [
    "difficulty_walking_steps","difficulty_walking_short","difficulty_walking_medium","difficulty_climbing_steps",
    "difficulty_using_hands","difficulty_raising_arm","difficulty_self_care","difficulty_communicating",
    "difficulty_remembering","difficulty_errands_alone","difficulty_social_activities",
]
CHRONIC_CONDITION_COLS = [
    "hypertension_ever","high_cholesterol_ever","heart_disease_ever","heart_attack_ever","angina_ever","stroke_ever",
    "asthma_ever","diabetes_ever","prediabetes_ever","cancer_ever","lung_disease_ever","arthritis_ever","dementia_ever","chronic_fatigue_ever",
]
COST_BARRIER_YES_COLS = [
    "trouble_paying_medical_bills","delayed_care_due_to_cost","skipped_care_due_to_cost",
    "skipped_medication_to_save_money","took_less_medication_to_save_money","delayed_prescription_to_save_money",
    "skipped_prescription_due_to_cost","delayed_therapy_due_to_cost","skipped_therapy_due_to_cost",
]
INCOME_SOURCE_COLS = ["has_wage_income","has_investment_income","has_social_security_income","has_disability_income","has_public_assistance_income","has_retirement_income","has_other_income"]
INSURANCE_YES_COLS = ["has_health_insurance","has_private_insurance","has_senior_gov_insurance","has_lowincome_gov_insurance","has_child_gov_insurance","has_military_insurance","has_state_health_plan","has_other_gov_insurance"]
FREQUENCY_COLS=[CFG.HOUSEHOLD_COL,"pseudo_stratum","pseudo_psu"]

# Conservative nominal list: these are not treated as ordinal just because their source codes are integers.
NATIVE_NOMINAL_COLS = [
    "record_type","survey_year","geographic_region","pseudo_stratum","pseudo_psu","sex","citizenship_status","born_in_country",
    "sexual_orientation","parental_status","relationship_to_child","marital_status","legal_marital_status","marital_status_detailed",
    "health_insurance_type","insurance_source_plan1","type_of_care_provider","home_ownership","smoking_status","ecigarette_status",
    "interview_quarter","interview_month",
]


def count_yes(df,cols):
    cols=[c for c in cols if c in df]
    return (df[cols]==1).sum(axis=1).astype(np.int16) if cols else pd.Series(0,index=df.index,dtype=np.int16)


def reverse_ordinal(series,max_valid):
    s=pd.to_numeric(series,errors="coerce")
    out=pd.Series(np.nan,index=s.index,dtype=float)
    valid=s.between(1,max_valid)
    out.loc[valid]=(max_valid+1)-s.loc[valid]
    return out


class SemanticFeatureEngineer:
    def __init__(self, valid_code_map):
        self.valid_code_map=valid_code_map
        self.frequency_maps_={}
        self.engineered_feature_names_=[]
        self.semantic_cols_=[]

    def fit(self,df):
        for c in FREQUENCY_COLS:
            if c in df:
                self.frequency_maps_[c]=df[c].value_counts(dropna=False).to_dict()
        return self

    def transform(self,df):
        out=df.copy(); before=set(out.columns)

        # Fold-safe frequency encoding only; household raw ID is later dropped.
        for c,mapping in self.frequency_maps_.items():
            if c in out:
                out[f"{c}__frequency"]=out[c].map(mapping).fillna(0).astype(np.float32)
                out[f"{c}__log_frequency"]=np.log1p(out[f"{c}__frequency"]).astype(np.float32)

        # V2 dual semantic representation for undocumented survey codes.
        semantic_cols=[]; nonstd_cols=[]
        for c in SEMANTIC_CODE_COLS:
            if c not in out or c not in self.valid_code_map:
                continue
            raw=pd.to_numeric(out[c],errors="coerce")
            valid=self.valid_code_map[c]
            sem_name=f"{c}__semantic"; nonstd_name=f"{c}__nonstandard"
            out[sem_name]=raw.where(raw.isin(valid)).astype(np.float32)
            out[nonstd_name]=(raw.notna() & ~raw.isin(valid)).astype(np.int8)
            semantic_cols.append(sem_name); nonstd_cols.append(nonstd_name)
        out["nonstandard_code_count"]=out[nonstd_cols].sum(axis=1).astype(np.int16) if nonstd_cols else 0
        self.semantic_cols_=semantic_cols

        # Curated domain burdens from V1.
        out["functional_difficulty_count"]=count_yes(out,FUNCTIONAL_DIFFICULTY_COLS)
        out["chronic_condition_count"]=count_yes(out,CHRONIC_CONDITION_COLS)
        out["cost_barrier_count"]=count_yes(out,COST_BARRIER_YES_COLS)
        out["income_source_count"]=count_yes(out,INCOME_SOURCE_COLS)
        out["insurance_count"]=count_yes(out,INSURANCE_YES_COLS)

        food=[]
        for c in ["worried_food_would_run_out","food_did_not_last","could_not_afford_balanced_meals"]:
            if c in out: food.append(reverse_ordinal(out[c],3).fillna(0))
        out["food_insecurity_count"]=(sum(food).astype(np.float32) if food else 0.0)
        if "received_food_assistance" in out:
            out["food_insecurity_count"] += (out["received_food_assistance"]==1).astype(np.float32)

        if "depression_frequency" in out:
            out["depression_burden"]=reverse_ordinal(out["depression_frequency"],5).astype(np.float32)
        if "anxiety_frequency" in out:
            out["anxiety_burden"]=reverse_ordinal(out["anxiety_frequency"],5).astype(np.float32)
        mental=[]
        for c in ["depression_burden","anxiety_burden"]:
            if c in out: mental.append(out[c].fillna(0))
        out["mental_frequency_burden"]=(sum(mental).astype(np.float32) if mental else 0.0)

        hh=[]
        for c in ["children_in_household","adults_in_household"]:
            if c in out: hh.append(pd.to_numeric(out[c],errors="coerce").fillna(0))
        out["household_size_proxy"]=(sum(hh).astype(np.float32) if hh else 0.0)

        # Conservative discretization only.
        if "age" in out:
            age=pd.to_numeric(out["age"],errors="coerce").where(pd.to_numeric(out["age"],errors="coerce").between(18,85))
            out["age_band"]=pd.cut(age,bins=[17,24,34,44,54,64,74,85],labels=False,include_lowest=True).astype(float)

        # Small semantic interaction bank: depth shortcuts, not feature explosion.
        def valid_product(a,b,name):
            if a in out and b in out:
                out[name]=(pd.to_numeric(out[a],errors="coerce")*pd.to_numeric(out[b],errors="coerce")).astype(np.float32)
        valid_product("general_health_status__semantic","depression_burden","health_x_depression_burden")
        valid_product("general_health_status__semantic","anxiety_burden","health_x_anxiety_burden")
        valid_product("income_poverty_category__semantic","food_security_4cat__semantic","poverty_x_food_security")
        valid_product("mental_frequency_burden","functional_difficulty_count","mental_x_functional_burden")

        # Never expose respondent/household identity to boosters.
        out=out.drop(columns=[c for c in [CFG.ID_COL,CFG.HOUSEHOLD_COL] if c in out],errors="ignore")
        self.engineered_feature_names_=[c for c in out.columns if c not in before]
        return out

    def fit_transform(self,df):
        return self.fit(df).transform(df)


In [8]:
DOMAIN_FORCE_KEEP=[
    "general_health_status","general_health_status__semantic",
    "depression_frequency","depression_frequency__semantic","depression_burden",
    "anxiety_frequency","anxiety_frequency__semantic","anxiety_burden","anxiety_level",
    "income_to_poverty_ratio","income_poverty_category","income_poverty_category__semantic",
    "food_security_4cat","food_security_4cat__semantic","food_security_3cat",
    "difficulty_social_activities","difficulty_remembering","marital_status","marital_status_detailed",
    "age","survey_weight","row_missing_count_raw","row_missing_count_after_repair","repaired_value_count",
    "special_code_count","nonstandard_code_count","functional_difficulty_count","chronic_condition_count",
    "cost_barrier_count","food_insecurity_count","income_source_count","insurance_count","household_size_proxy",
    "mental_frequency_burden","health_x_depression_burden","health_x_anxiety_burden","poverty_x_food_security","mental_x_functional_burden",
]


In [9]:
def make_lgb_fs(seed, use_gpu=True):
    params=dict(
        objective="regression_l2",n_estimators=CFG.INNER_FS_ESTIMATORS,learning_rate=0.035,
        num_leaves=31,min_child_samples=40,colsample_bytree=0.9,subsample=0.9,subsample_freq=1,
        reg_lambda=4.0,random_state=seed,n_jobs=-1,verbosity=-1,
    )
    if use_gpu: params.update(device_type="gpu",max_bin=63)
    return lgb.LGBMRegressor(**params)


def nested_select_features(X_outer_train,y_outer_train,groups_outer_train,engineered_names,seed):
    y_outer_train=np.asarray(y_outer_train)
    inner_strata=pd.Series(y_outer_train).astype(str).values
    inner_cv=StratifiedGroupKFold(n_splits=CFG.INNER_N_FOLDS,shuffle=True,random_state=seed)
    i_tr,i_va=next(inner_cv.split(X_outer_train,inner_strata,groups=groups_outer_train))
    all_features=X_outer_train.columns.tolist()

    model=None
    for gpu_flag in ([True,False] if CFG.USE_GPU else [False]):
        try:
            model=make_lgb_fs(seed,use_gpu=gpu_flag)
            model.fit(X_outer_train.iloc[i_tr],y_outer_train[i_tr],eval_set=[(X_outer_train.iloc[i_va],y_outer_train[i_va])],
                      callbacks=[lgb.early_stopping(CFG.INNER_FS_EARLY_STOPPING,verbose=False)])
            break
        except Exception as e:
            if gpu_flag:
                print("Inner FS LightGBM GPU fallback:",repr(e)); gc.collect(); continue
            raise

    gain=model.booster_.feature_importance(importance_type="gain")
    ranked=[f for _,f in sorted(zip(gain,all_features),reverse=True)]
    mandatory=[c for c in DOMAIN_FORCE_KEEP if c in X_outer_train]
    mandatory += [c for c in engineered_names if c in X_outer_train]
    mandatory += [c for c in X_outer_train if c.endswith("__nonstandard") or c.endswith("__special_code")]
    selected=list(dict.fromkeys(ranked[:CFG.INNER_TOP_N]+mandatory))
    return selected, pd.DataFrame({"feature":all_features,"gain":gain}).sort_values("gain",ascending=False)


In [10]:
def prepare_catboost_frame(X,features):
    Xc=X[features].copy()
    cat_cols=[c for c in NATIVE_NOMINAL_COLS if c in Xc]
    for c in cat_cols:
        Xc[c]=Xc[c].map(lambda v:"__MISSING__" if pd.isna(v) else str(v))
    return Xc,cat_cols


def prepare_native_categorical_views(X_tr,X_va,X_te,features):
    A=X_tr[features].copy(); B=X_va[features].copy(); C=X_te[features].copy()
    cat_cols=[]
    for c in NATIVE_NOMINAL_COLS:
        if c not in A:
            continue
        source=f"{c}__semantic" if f"{c}__semantic" in A else c
        if source not in A:
            continue
        new=f"{c}__cat"
        combined=pd.concat([A[source],B[source],C[source]],axis=0)
        levels=pd.Index(combined.dropna().astype(str).unique())
        for frame,src in [(A,source),(B,source),(C,source)]:
            vals=frame[src].map(lambda v:"__MISSING__" if pd.isna(v) else str(v))
            frame[new]=pd.Categorical(vals,categories=list(levels)+(["__MISSING__"] if "__MISSING__" not in levels else []))
        cat_cols.append(new)
    return A,B,C,cat_cols


def monotone_constraint_vector(features):
    direction={
        "general_health_status__semantic":+1,
        "depression_burden":+1,
        "anxiety_burden":+1,
        "mental_frequency_burden":+1,
        "functional_difficulty_count":+1,
        "food_insecurity_count":+1,
        "cost_barrier_count":+1,
        "income_to_poverty_ratio":-1,
        "income_poverty_category__semantic":-1,
        "food_security_4cat__semantic":+1,
        "worry_about_medical_costs__semantic":-1,
    }
    return [int(direction.get(c,0)) for c in features]


In [11]:
def make_lgb_reg(seed,use_gpu=True,monotone_constraints=None):
    params=dict(
        objective="regression_l2",n_estimators=CFG.LGB_N_ESTIMATORS,learning_rate=CFG.LEARNING_RATE,
        num_leaves=31,min_child_samples=40,max_depth=-1,subsample=0.85,subsample_freq=1,colsample_bytree=0.85,
        reg_alpha=0.05,reg_lambda=4.0,random_state=seed,n_jobs=-1,verbosity=-1,
    )
    if monotone_constraints is not None:
        params["monotone_constraints"]=monotone_constraints
        params["monotone_constraints_method"]="advanced"
    if use_gpu: params.update(device_type="gpu",max_bin=63)
    return lgb.LGBMRegressor(**params)


def make_lgb_classifier(seed,objective="multiclass",use_gpu=True):
    params=dict(
        n_estimators=CFG.LGB_N_ESTIMATORS,learning_rate=CFG.LEARNING_RATE,num_leaves=31,min_child_samples=40,
        subsample=0.85,subsample_freq=1,colsample_bytree=0.85,reg_lambda=4.0,random_state=seed,n_jobs=-1,verbosity=-1,
    )
    if objective=="multiclass": params.update(objective="multiclass",num_class=4)
    else: params.update(objective="binary")
    if use_gpu: params.update(device_type="gpu",max_bin=63)
    return lgb.LGBMClassifier(**params)


def make_xgb_reg(seed,use_gpu=True,enable_categorical=False):
    params=dict(
        objective="reg:squarederror",eval_metric="rmse",n_estimators=CFG.XGB_N_ESTIMATORS,learning_rate=CFG.LEARNING_RATE,
        max_depth=6,min_child_weight=8,subsample=0.85,colsample_bytree=0.85,reg_alpha=0.05,reg_lambda=4.0,
        tree_method="hist",random_state=seed,n_jobs=-1,early_stopping_rounds=CFG.EARLY_STOPPING,
        enable_categorical=enable_categorical,
    )
    if enable_categorical:
        params["max_cat_to_onehot"]=4
        params["max_cat_threshold"]=64
    if use_gpu: params["device"]="cuda"
    return xgb.XGBRegressor(**params)


def make_cat(seed,use_gpu=True):
    params=dict(
        loss_function="RMSE",eval_metric="RMSE",iterations=CFG.CAT_ITERATIONS,learning_rate=CFG.LEARNING_RATE,depth=7,
        l2_leaf_reg=6.0,random_strength=0.5,random_seed=seed,od_type="Iter",od_wait=CFG.EARLY_STOPPING,
        allow_writing_files=False,verbose=CFG.VERBOSE,
    )
    if use_gpu: params.update(task_type="GPU",devices=CFG.CAT_DEVICES)
    else: params.update(task_type="CPU",thread_count=-1)
    return CatBoostRegressor(**params)


def fit_lgb_reg(Xtr,ytr,Xva,yva,Xte,seed,monotone=None,categorical_feature=None):
    for gpu in ([True,False] if CFG.USE_GPU else [False]):
        try:
            m=make_lgb_reg(seed,gpu,monotone)
            m.fit(Xtr,ytr,eval_set=[(Xva,yva)],eval_metric="rmse",categorical_feature=categorical_feature,
                  callbacks=[lgb.early_stopping(CFG.EARLY_STOPPING,verbose=False),lgb.log_evaluation(CFG.VERBOSE)])
            return m,m.predict(Xva,num_iteration=m.best_iteration_),m.predict(Xte,num_iteration=m.best_iteration_)
        except Exception as e:
            if gpu:
                print("LightGBM GPU -> CPU fallback:",repr(e)); gc.collect(); continue
            raise


def fit_xgb_reg(Xtr,ytr,Xva,yva,Xte,seed,enable_categorical=False):
    for gpu in ([True,False] if CFG.USE_GPU else [False]):
        try:
            m=make_xgb_reg(seed,gpu,enable_categorical)
            m.fit(Xtr,ytr,eval_set=[(Xva,yva)],verbose=False)
            return m,m.predict(Xva),m.predict(Xte)
        except Exception as e:
            if gpu:
                print("XGBoost CUDA -> CPU fallback:",repr(e)); gc.collect(); continue
            raise


def fit_cat(Xtr,ytr,Xva,yva,Xte,cat_cols,seed):
    for gpu in ([True,False] if CFG.USE_GPU else [False]):
        try:
            m=make_cat(seed,gpu)
            m.fit(Xtr,ytr,eval_set=(Xva,yva),cat_features=cat_cols,use_best_model=True)
            return m,m.predict(Xva),m.predict(Xte)
        except Exception as e:
            if gpu:
                print("CatBoost GPU -> CPU fallback:",repr(e)); gc.collect(); continue
            raise


def fit_lgb_multiclass_expected(Xtr,ytr,Xva,yva,Xte,seed):
    y0=np.asarray(ytr)-1; yv0=np.asarray(yva)-1
    for gpu in ([True,False] if CFG.USE_GPU else [False]):
        try:
            m=make_lgb_classifier(seed,"multiclass",gpu)
            m.fit(Xtr,y0,eval_set=[(Xva,yv0)],eval_metric="multi_logloss",
                  callbacks=[lgb.early_stopping(CFG.EARLY_STOPPING,verbose=False),lgb.log_evaluation(CFG.VERBOSE)])
            pv=m.predict_proba(Xva,num_iteration=m.best_iteration_)
            pt=m.predict_proba(Xte,num_iteration=m.best_iteration_)
            classes=np.arange(1,5,dtype=float)
            return m,pv@classes,pt@classes
        except Exception as e:
            if gpu:
                print("LGB multiclass GPU -> CPU fallback:",repr(e)); gc.collect(); continue
            raise


def fit_lgb_cumulative_ordinal(Xtr,ytr,Xva,yva,Xte,seed):
    pva=[]; pte=[]; models=[]
    for boundary in [1,2,3]:
        yt=(np.asarray(ytr)>boundary).astype(int); yv=(np.asarray(yva)>boundary).astype(int)
        fitted=None
        for gpu in ([True,False] if CFG.USE_GPU else [False]):
            try:
                m=make_lgb_classifier(seed+boundary*101,"binary",gpu)
                m.fit(Xtr,yt,eval_set=[(Xva,yv)],eval_metric="binary_logloss",
                      callbacks=[lgb.early_stopping(CFG.EARLY_STOPPING,verbose=False)])
                pva.append(m.predict_proba(Xva,num_iteration=m.best_iteration_)[:,1])
                pte.append(m.predict_proba(Xte,num_iteration=m.best_iteration_)[:,1])
                models.append(m); fitted=True; break
            except Exception as e:
                if gpu:
                    print(f"Ordinal boundary {boundary} GPU -> CPU fallback:",repr(e)); gc.collect(); continue
                raise
        assert fitted
    pva=np.column_stack(pva); pte=np.column_stack(pte)
    # Enforce cumulative consistency P(y>1)>=P(y>2)>=P(y>3).
    pva[:,1]=np.minimum(pva[:,1],pva[:,0]); pva[:,2]=np.minimum(pva[:,2],pva[:,1])
    pte[:,1]=np.minimum(pte[:,1],pte[:,0]); pte[:,2]=np.minimum(pte[:,2],pte[:,1])
    return models,1.0+pva.sum(axis=1),1.0+pte.sum(axis=1)


### 2.1 Execute exact V2 repeated-CV branches

GPU-to-CPU fallback here is inherited from historical V2 and affects only the **control**. V9 foundation inference itself has no model fallback.


In [12]:
branch_names=["lgb_core","xgb_core","cat_core"]
if CFG.RUN_LGB_CATEGORICAL: branch_names.append("lgb_categorical")
if CFG.RUN_XGB_CATEGORICAL: branch_names.append("xgb_categorical")
if CFG.RUN_MONOTONIC: branch_names.append("lgb_monotonic")
if CFG.RUN_MULTICLASS: branch_names.append("lgb_multiclass_expected")
if CFG.RUN_ORDINAL: branch_names.append("lgb_cumulative_ordinal")

OOF_SUM={b:np.zeros(len(train),dtype=np.float64) for b in branch_names}
OOF_COUNT={b:np.zeros(len(train),dtype=np.int8) for b in branch_names}
TEST_SUM={b:np.zeros(len(test),dtype=np.float64) for b in branch_names}
branch_failures=defaultdict(list)
fold_metric_rows=[]
feature_selection_rows=[]
feature_importance_rows=[]
primary_fold_id=np.full(len(train),-1,dtype=np.int8)

for repeat,fold,seed,tr_idx,va_idx in repeated_folds:
    print("\n"+"="*90)
    print(f"REPEAT {repeat+1}/{CFG.N_REPEATS} | FOLD {fold+1}/{CFG.N_FOLDS} | seed={seed}")
    print("="*90)
    if repeat==0: primary_fold_id[va_idx]=fold

    raw_tr=train.iloc[tr_idx].copy(); raw_va=train.iloc[va_idx].copy(); raw_te=test.copy()
    ytr=y[tr_idx]; yva=y[va_idx]; gtr=groups[tr_idx]

    # Fold-safe preprocessing and semantic FE.
    pre=FoldPreprocessor(CFG.REPAIR_MIN_ACCURACY,CFG.REPAIR_MIN_SUPPORT).fit(raw_tr)
    ptr=pre.transform(raw_tr); pva=pre.transform(raw_va); pte=pre.transform(raw_te)
    fe=SemanticFeatureEngineer(DICT_VALID_CODES).fit(ptr)
    Xtr=fe.transform(ptr); Xva=fe.transform(pva); Xte=fe.transform(pte)

    assert Xtr.columns.tolist()==Xva.columns.tolist()==Xte.columns.tolist()
    selected,fs_imp=nested_select_features(Xtr,ytr,gtr,fe.engineered_feature_names_,seed+fold*31)
    feature_selection_rows.append({"repeat":repeat,"fold":fold,"selected_count":len(selected),"selected_features":"|".join(selected)})

    A=Xtr[selected].copy(); B=Xva[selected].copy(); C=Xte[selected].copy()
    model_seed=seed+fold*997

    # --- Core LightGBM regression ---
    m,pv,pt=fit_lgb_reg(A,ytr,B,yva,C,model_seed)
    OOF_SUM["lgb_core"][va_idx]+=pv; OOF_COUNT["lgb_core"][va_idx]+=1; TEST_SUM["lgb_core"]+=pt
    for f,imp in zip(selected,m.booster_.feature_importance(importance_type="gain")):
        feature_importance_rows.append({"repeat":repeat,"fold":fold,"branch":"lgb_core","feature":f,"importance":float(imp)})

    # --- Core XGBoost regression ---
    m,pv,pt=fit_xgb_reg(A,ytr,B,yva,C,model_seed+11,False)
    OOF_SUM["xgb_core"][va_idx]+=pv; OOF_COUNT["xgb_core"][va_idx]+=1; TEST_SUM["xgb_core"]+=pt
    for f,imp in zip(selected,m.feature_importances_):
        feature_importance_rows.append({"repeat":repeat,"fold":fold,"branch":"xgb_core","feature":f,"importance":float(imp)})

    # --- Core CatBoost regression ---
    Act,cat_cols=prepare_catboost_frame(Xtr,selected); Bct,_=prepare_catboost_frame(Xva,selected); Cct,_=prepare_catboost_frame(Xte,selected)
    m,pv,pt=fit_cat(Act,ytr,Bct,yva,Cct,cat_cols,model_seed+23)
    OOF_SUM["cat_core"][va_idx]+=pv; OOF_COUNT["cat_core"][va_idx]+=1; TEST_SUM["cat_core"]+=pt
    for f,imp in zip(selected,m.get_feature_importance()):
        feature_importance_rows.append({"repeat":repeat,"fold":fold,"branch":"cat_core","feature":f,"importance":float(imp)})

    # --- Native categorical branches ---
    if CFG.RUN_LGB_CATEGORICAL or CFG.RUN_XGB_CATEGORICAL:
        Ac,Bc,Cc,native_cat_cols=prepare_native_categorical_views(Xtr,Xva,Xte,selected)
        native_features=Ac.columns.tolist()

        if CFG.RUN_LGB_CATEGORICAL:
            try:
                m,pv,pt=fit_lgb_reg(Ac,ytr,Bc,yva,Cc,model_seed+37,categorical_feature=native_cat_cols)
                OOF_SUM["lgb_categorical"][va_idx]+=pv; OOF_COUNT["lgb_categorical"][va_idx]+=1; TEST_SUM["lgb_categorical"]+=pt
            except Exception as e:
                branch_failures["lgb_categorical"].append((repeat,fold,repr(e))); print("SKIP lgb_categorical fold:",repr(e))

        if CFG.RUN_XGB_CATEGORICAL:
            try:
                m,pv,pt=fit_xgb_reg(Ac,ytr,Bc,yva,Cc,model_seed+41,True)
                OOF_SUM["xgb_categorical"][va_idx]+=pv; OOF_COUNT["xgb_categorical"][va_idx]+=1; TEST_SUM["xgb_categorical"]+=pt
            except Exception as e:
                branch_failures["xgb_categorical"].append((repeat,fold,repr(e))); print("SKIP xgb_categorical fold:",repr(e))

    # --- Monotonic semantic branch ---
    if CFG.RUN_MONOTONIC:
        try:
            mono=monotone_constraint_vector(selected)
            m,pv,pt=fit_lgb_reg(A,ytr,B,yva,C,model_seed+53,monotone=mono)
            OOF_SUM["lgb_monotonic"][va_idx]+=pv; OOF_COUNT["lgb_monotonic"][va_idx]+=1; TEST_SUM["lgb_monotonic"]+=pt
        except Exception as e:
            branch_failures["lgb_monotonic"].append((repeat,fold,repr(e))); print("SKIP lgb_monotonic fold:",repr(e))

    # --- Objective diversity sidecars ---
    if CFG.RUN_MULTICLASS:
        try:
            m,pv,pt=fit_lgb_multiclass_expected(A,ytr,B,yva,C,model_seed+67)
            OOF_SUM["lgb_multiclass_expected"][va_idx]+=pv; OOF_COUNT["lgb_multiclass_expected"][va_idx]+=1; TEST_SUM["lgb_multiclass_expected"]+=pt
        except Exception as e:
            branch_failures["lgb_multiclass_expected"].append((repeat,fold,repr(e))); print("SKIP multiclass fold:",repr(e))

    if CFG.RUN_ORDINAL:
        try:
            _,pv,pt=fit_lgb_cumulative_ordinal(A,ytr,B,yva,C,model_seed+79)
            OOF_SUM["lgb_cumulative_ordinal"][va_idx]+=pv; OOF_COUNT["lgb_cumulative_ordinal"][va_idx]+=1; TEST_SUM["lgb_cumulative_ordinal"]+=pt
        except Exception as e:
            branch_failures["lgb_cumulative_ordinal"].append((repeat,fold,repr(e))); print("SKIP ordinal fold:",repr(e))

    gc.collect()

assert (primary_fold_id>=0).all()
print("\nBranch failures:")
for b,items in branch_failures.items(): print(b,len(items),items[:2])



REPEAT 1/2 | FOLD 1/5 | seed=42


/tmp/ipykernel_23/2635968242.py:72: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[nonstd_name]=(raw.notna() & ~raw.isin(valid)).astype(np.int8)
/tmp/ipykernel_23/2635968242.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[sem_name]=raw.where(raw.isin(valid)).astype(np.float32)
/tmp/ipykernel_23/2635968242.py:72: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) 

[400]	valid_0's rmse: 0.513723	valid_0's l2: 0.263912


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [19:36:20] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


0:	learn: 0.6040931	test: 0.6053679	best: 0.6053679 (0)	total: 133ms	remaining: 12m 10s
400:	learn: 0.5098318	test: 0.5147950	best: 0.5147921 (399)	total: 9.44s	remaining: 2m
800:	learn: 0.5022121	test: 0.5138868	best: 0.5138868 (800)	total: 18.8s	remaining: 1m 50s
1200:	learn: 0.4959806	test: 0.5134682	best: 0.5134546 (1184)	total: 28.3s	remaining: 1m 41s
1600:	learn: 0.4900322	test: 0.5132918	best: 0.5132873 (1576)	total: 37.8s	remaining: 1m 32s
bestTest = 0.5132066208
bestIteration = 1778
Shrink model to first 1779 iterations.


...

[400]	valid_0's rmse: 0.516633	valid_0's l2: 0.26691
[800]	valid_0's rmse: 0.516254	valid_0's l2: 0.266518
0:	learn: 0.6038635	test: 0.6063468	best: 0.6063468 (0)	total: 33.9ms	remaining: 3m 6s
400:	learn: 0.5092365	test: 0.5181964	best: 0.5181964 (400)	total: 9.48s	remaining: 2m
800:	learn: 0.5016274	test: 0.5170599	best: 0.5170565 (799)	total: 18.7s	remaining: 1m 49s
1200:	learn: 0.4953276	test: 0.5167091	best: 0.5167080 (1196)	total: 28.1s	remaining: 1m 40s
1600:	learn: 0.4893864	test: 0.5164531	best: 0.5164353 (1498)	total: 37.5s	remaining: 1m 31s
2000:	learn: 0.4837379	test: 0.5163165	best: 0.5163165 (2000)	total: 47s	remaining: 1m 22s
2400:	learn: 0.4780419	test: 0.5162055	best: 0.5161985 (2392)	total: 56.7s	remaining: 1m 13s
bestTest = 0.5161601366
bestIteration = 2526
Shrink model to first 2527 iterations.
[400]	valid_0's rmse: 0.518584	valid_0's l2: 0.26893
[400]	valid_0's rmse: 0.516628	valid_0's l2: 0.266905
[800]	valid_0's rmse: 0.516374	valid_0's l2: 0.266642
[400]	valid_0

/tmp/ipykernel_23/2635968242.py:72: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[nonstd_name]=(raw.notna() & ~raw.isin(valid)).astype(np.int8)
/tmp/ipykernel_23/2635968242.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[sem_name]=raw.where(raw.isin(valid)).astype(np.float32)
/tmp/ipykernel_23/2635968242.py:72: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) 

[400]	valid_0's rmse: 0.518795	valid_0's l2: 0.269148
[800]	valid_0's rmse: 0.518537	valid_0's l2: 0.26888
0:	learn: 0.6034173	test: 0.6080112	best: 0.6080112 (0)	total: 35.3ms	remaining: 3m 14s
400:	learn: 0.5089433	test: 0.5202115	best: 0.5202115 (400)	total: 9.81s	remaining: 2m 4s
800:	learn: 0.5016196	test: 0.5191277	best: 0.5191277 (800)	total: 19.6s	remaining: 1m 55s
1200:	learn: 0.4955778	test: 0.5187100	best: 0.5187093 (1198)	total: 29.6s	remaining: 1m 46s
1600:	learn: 0.4899080	test: 0.5184768	best: 0.5184719 (1594)	total: 39.6s	remaining: 1m 36s
2000:	learn: 0.4843291	test: 0.5182811	best: 0.5182810 (1998)	total: 49.6s	remaining: 1m 26s
bestTest = 0.5182616637
bestIteration = 2021
Shrink model to first 2022 iterations.
[400]	valid_0's rmse: 0.520876	valid_0's l2: 0.271312
[400]	valid_0's rmse: 0.518872	valid_0's l2: 0.269228
[800]	valid_0's rmse: 0.51858	valid_0's l2: 0.268925
[400]	valid_0's multi_logloss: 0.708216

REPEAT 2/2 | FOLD 5/5 | seed=2026


/tmp/ipykernel_23/2635968242.py:72: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[nonstd_name]=(raw.notna() & ~raw.isin(valid)).astype(np.int8)
/tmp/ipykernel_23/2635968242.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[sem_name]=raw.where(raw.isin(valid)).astype(np.float32)
/tmp/ipykernel_23/2635968242.py:72: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) 

[400]	valid_0's rmse: 0.515802	valid_0's l2: 0.266052
[800]	valid_0's rmse: 0.51532	valid_0's l2: 0.265555
0:	learn: 0.6056495	test: 0.5988586	best: 0.5988586 (0)	total: 38.7ms	remaining: 3m 32s
400:	learn: 0.5096637	test: 0.5172688	best: 0.5172686 (399)	total: 11.3s	remaining: 2m 23s
800:	learn: 0.5020684	test: 0.5160579	best: 0.5160579 (800)	total: 22.3s	remaining: 2m 10s
1200:	learn: 0.4960971	test: 0.5155423	best: 0.5155423 (1200)	total: 33.6s	remaining: 2m
1600:	learn: 0.4903242	test: 0.5151497	best: 0.5151490 (1598)	total: 45.1s	remaining: 1m 49s
2000:	learn: 0.4844543	test: 0.5150725	best: 0.5150700 (1957)	total: 56.3s	remaining: 1m 38s
bestTest = 0.5150110368
bestIteration = 2125
Shrink model to first 2126 iterations.
[400]	valid_0's rmse: 0.51783	valid_0's l2: 0.268148
[800]	valid_0's rmse: 0.517874	valid_0's l2: 0.268194
[400]	valid_0's rmse: 0.515602	valid_0's l2: 0.265846
[800]	valid_0's rmse: 0.515361	valid_0's l2: 0.265597
[400]	valid_0's multi_logloss: 0.703067

Branch f

In [13]:
# Keep only branches that produced exactly one OOF prediction per repeat for every row.
valid_branches=[]; oof_cont={}; test_cont={}
for b in branch_names:
    complete=np.all(OOF_COUNT[b]==CFG.N_REPEATS)
    if complete:
        valid_branches.append(b)
        oof_cont[b]=OOF_SUM[b]/CFG.N_REPEATS
        test_cont[b]=TEST_SUM[b]/(CFG.N_REPEATS*CFG.N_FOLDS)
    else:
        print(f"Dropping incomplete branch {b}: OOF count range {OOF_COUNT[b].min()}..{OOF_COUNT[b].max()}")

print("Valid complete branches:",valid_branches)
assert all(x in valid_branches for x in ["lgb_core","xgb_core","cat_core"])


def crossfit_threshold_score(y_true,pred,fold_ids):
    y_true=np.asarray(y_true); pred=np.asarray(pred); fold_ids=np.asarray(fold_ids)
    labels=np.zeros(len(y_true),dtype=int); thresholds={}
    for f in sorted(np.unique(fold_ids)):
        tr=fold_ids!=f; va=fold_ids==f
        t,_=optimize_thresholds(y_true[tr],pred[tr],maxiter=130)
        labels[va]=apply_thresholds(pred[va],t); thresholds[int(f)]=t
    return qwk(y_true,labels),labels,thresholds

branch_eval=[]
for b in valid_branches:
    p=oof_cont[b]
    t,g=optimize_thresholds(y,p)
    cf,lab,cft=crossfit_threshold_score(y,p,primary_fold_id)
    auc1=roc_auc_score((y>1).astype(int),p)
    auc2=roc_auc_score((y>2).astype(int),p)
    auc3=roc_auc_score((y>3).astype(int),p)
    branch_eval.append({"branch":b,"rmse":mean_squared_error(y,p)**0.5,"global_qwk":g,"crossfit_qwk":cf,"auc_y_gt_1":auc1,"auc_y_gt_2":auc2,"auc_y_gt_3":auc3,"t1":t[0],"t2":t[1],"t3":t[2]})
branch_eval_df=pd.DataFrame(branch_eval).sort_values("crossfit_qwk",ascending=False)
display(branch_eval_df)


Valid complete branches: ['lgb_core', 'xgb_core', 'cat_core', 'lgb_categorical', 'xgb_categorical', 'lgb_monotonic', 'lgb_multiclass_expected', 'lgb_cumulative_ordinal']


,branch,rmse,global_qwk,crossfit_qwk,auc_y_gt_1,auc_y_gt_2,auc_y_gt_3,t1,t2,t3
2,cat_core,0.515949,0.481384,0.480299,0.779387,0.871904,0.808949,1.557347,2.030081,2.524685
1,xgb_core,0.516148,0.479372,0.478631,0.778890,0.871581,0.807369,1.568498,2.037546,2.484152
5,lgb_monotonic,0.516078,0.480028,0.478494,0.778931,0.871347,0.808241,1.565550,2.014988,2.568232
7,lgb_cumulative_ordinal,0.515725,0.480584,0.478350,0.779796,0.871713,0.807809,1.565776,2.010908,2.486037
0,lgb_core,0.516046,0.479447,0.477848,0.778865,0.871783,0.808292,1.554390,2.010827,2.566504
6,lgb_multiclass_expected,0.516285,0.479310,0.476622,0.779608,0.870314,0.805843,1.582257,2.008512,2.485141
4,xgb_categorical,0.519011,0.475834,0.474813,0.775405,0.867853,0.801784,1.560248,2.032145,2.575328
3,lgb_categorical,0.518243,0.475209,0.474303,0.776011,0.869228,0.804204,1.544805,2.018314,2.583702


In [14]:
core_names=["lgb_core","xgb_core","cat_core"]
core_oof=np.mean(np.column_stack([oof_cont[b] for b in core_names]),axis=1)
core_test=np.mean(np.column_stack([test_cont[b] for b in core_names]),axis=1)
core_t,core_global=optimize_thresholds(y,core_oof)
core_cross,core_cross_labels,core_cross_t=crossfit_threshold_score(y,core_oof,primary_fold_id)
print("V2 repeated-CV core global QWK:",round(core_global,6))
print("V2 repeated-CV core cross-fitted QWK:",round(core_cross,6))
print("Historical V1 cross-fitted OOF:",CFG.V1_CROSSFIT_OOF)


V2 repeated-CV core global QWK: 0.48106
V2 repeated-CV core cross-fitted QWK: 0.479307
Historical V1 cross-fitted OOF: 0.479662


In [15]:
current_oof=core_oof.copy(); current_test=core_test.copy()
current_score=core_cross
selected_steps=[]
remaining=[b for b in valid_branches if b not in core_names]

for step in range(CFG.MAX_SELECTED_SIDECARS):
    best=None
    for b in remaining:
        for alpha in CFG.GREEDY_ALPHA_GRID:
            cand=(1-alpha)*current_oof+alpha*oof_cont[b]
            score,_,_=crossfit_threshold_score(y,cand,primary_fold_id)
            rec={"branch":b,"alpha":float(alpha),"score":float(score),"gain":float(score-current_score),
                 "residual_corr_to_current":float(np.corrcoef(y-current_oof,y-oof_cont[b])[0,1])}
            if best is None or rec["score"]>best["score"]:
                best=rec
    if best is None or best["gain"]<CFG.MIN_BRANCH_BLEND_GAIN:
        print("Stop greedy selection. Best remaining gain:", None if best is None else best["gain"])
        break
    b=best["branch"]; a=best["alpha"]
    current_oof=(1-a)*current_oof+a*oof_cont[b]
    current_test=(1-a)*current_test+a*test_cont[b]
    current_score=best["score"]
    selected_steps.append(best)
    remaining.remove(b)
    print("Accepted:",best)

display(pd.DataFrame(selected_steps))

final_oof_cont=current_oof
final_test_cont_raw=current_test
final_thresholds,final_global_qwk=optimize_thresholds(y,final_oof_cont)
final_cross_qwk,final_cross_labels,final_cross_thresholds=crossfit_threshold_score(y,final_oof_cont,primary_fold_id)
print("Final global OOF QWK:",round(final_global_qwk,6))
print("Final cross-fitted OOF QWK:",round(final_cross_qwk,6))
print("Final thresholds:",np.round(final_thresholds,6))
print("Gain over repeated-CV core:",round(final_cross_qwk-core_cross,6))
print("Gain over historical V1 crossfit:",round(final_cross_qwk-CFG.V1_CROSSFIT_OOF,6))


Accepted: {'branch': 'lgb_multiclass_expected', 'alpha': 0.45, 'score': 0.4810680499943676, 'gain': 0.0017609359625633259, 'residual_corr_to_current': 0.9968589404139795}
Stop greedy selection. Best remaining gain: -0.0004233972343142822


,branch,alpha,score,gain,residual_corr_to_current
0,lgb_multiclass_expected,0.45,0.481068,0.001761,0.996859


Final global OOF QWK: 0.481551
Final cross-fitted OOF QWK: 0.481068
Final thresholds: [1.559139 2.004014 2.517161]
Gain over repeated-CV core: 0.001761
Gain over historical V1 crossfit: 0.001406


### 2.2 Convert V2 to evaluator-aligned unique-respondent control


In [16]:
# Collapse the frozen historical V2 predictions only at evaluation/export level.
anchor_oof_row = pd.DataFrame({
    CFG.ID_COL: train[CFG.ID_COL].values,
    CFG.TARGET: y,
    "fold": primary_fold_id,
    "v2_score": final_oof_cont,
})

# Because duplicate respondents are exact and GroupKFold-safe, target/fold must agree within respondent.
assert (anchor_oof_row.groupby(CFG.ID_COL)[CFG.TARGET].nunique() == 1).all()
assert (anchor_oof_row.groupby(CFG.ID_COL)["fold"].nunique() == 1).all()

v2_uid_oof = (
    anchor_oof_row.groupby(CFG.ID_COL, sort=False, as_index=False)
    .agg({CFG.TARGET: "first", "fold": "first", "v2_score": "mean"})
)

anchor_test_row = pd.DataFrame({CFG.ID_COL: test[CFG.ID_COL].values, "v2_score": final_test_cont_raw})
v2_uid_test = anchor_test_row.groupby(CFG.ID_COL, sort=False, as_index=False).v2_score.mean()

uid_y = v2_uid_oof[CFG.TARGET].astype(int).to_numpy()
uid_fold = v2_uid_oof["fold"].astype(int).to_numpy()
uid_v2 = v2_uid_oof["v2_score"].astype(float).to_numpy()

v2_uid_cf_qwk, v2_uid_cf_labels, v2_uid_fold_thresholds = crossfit_threshold_score(uid_y, uid_v2, uid_fold)
v2_uid_auc_gt1 = roc_auc_score((uid_y > 1).astype(int), uid_v2)
mask12 = np.isin(uid_y, [1, 2])
v2_uid_auc_1v2 = roc_auc_score((uid_y[mask12] == 2).astype(int), uid_v2[mask12])

assert CFG.V2_SANITY_QWK_MIN <= v2_uid_cf_qwk <= CFG.V2_SANITY_QWK_MAX, (
    f"Frozen V2 control QWK {v2_uid_cf_qwk:.6f} outside sanity range; stop before TabICL"
)
assert v2_uid_auc_gt1 >= CFG.V2_SANITY_AUC_GT1_MIN, (
    f"Frozen V2 boundary AUC {v2_uid_auc_gt1:.6f} unexpectedly weak; stop before TabICL"
)

v2_uid_oof["v2_crossfit_class"] = v2_uid_cf_labels
v2_uid_oof.to_csv(CFG.OUTPUT_DIR / "v2_anchor_unique_oof.csv", index=False)
v2_uid_test.to_csv(CFG.OUTPUT_DIR / "v2_anchor_unique_test.csv", index=False)

V2_ANCHOR_SUMMARY = {
    "row_level_historical_crossfit_qwk_this_run": float(final_cross_qwk),
    "unique_id_crossfit_qwk_this_run": float(v2_uid_cf_qwk),
    "unique_id_auc_gt1": float(v2_uid_auc_gt1),
    "unique_id_auc_1v2": float(v2_uid_auc_1v2),
    "audit_reference_unique_id_qwk": CFG.V2_EVALUATOR_ALIGNED_AUDIT_REFERENCE,
    "public_lb_reference": CFG.V2_PUBLIC_LB_REFERENCE,
    "selected_sidecars_this_run": selected_steps,
    "note": "Historical V2 sidecar selection is frozen control logic and is not claimed to be fully nested model-selection CV.",
}
with open(CFG.OUTPUT_DIR / "v2_anchor_summary.json", "w") as f:
    json.dump(V2_ANCHOR_SUMMARY, f, indent=2)
print(json.dumps(V2_ANCHOR_SUMMARY, indent=2))


{
  "row_level_historical_crossfit_qwk_this_run": 0.4810680499943676,
  "unique_id_crossfit_qwk_this_run": 0.4794660420797272,
  "unique_id_auc_gt1": 0.7797972758044079,
  "unique_id_auc_1v2": 0.7656628691557714,
  "audit_reference_unique_id_qwk": 0.48062,
  "public_lb_reference": 0.48193,
  "selected_sidecars_this_run": [
    {
      "branch": "lgb_multiclass_expected",
      "alpha": 0.45,
      "score": 0.4810680499943676,
      "gain": 0.0017609359625633259,
      "residual_corr_to_current": 0.9968589404139795
    }
  ],
  "note": "Historical V2 sidecar selection is frozen control logic and is not claimed to be fully nested model-selection CV."
}


## 3. Install and pin TabICLv2 runtime after the V2 control is frozen

The notebook requires internet access for the package/checkpoint unless the exact artifacts are already cached. Failure here is fatal; there is no fallback candidate.


In [17]:
def pip_install_exact(spec):
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", spec]
    print("Installing:", " ".join(cmd))
    try:
        subprocess.check_call(cmd)
    except Exception as e:
        raise RuntimeError(
            f"Dependency installation failed for {spec}. Enable Kaggle Internet or attach the exact package artifact."
        ) from e
    importlib.invalidate_caches()

pip_install_exact(f"tabicl=={CFG.TABICL_VERSION}")

import importlib.metadata as importlib_metadata
import torch
from huggingface_hub import hf_hub_download
from tabicl import TabICLRegressor

installed_tabicl = importlib_metadata.version("tabicl")
assert installed_tabicl == CFG.TABICL_VERSION, (installed_tabicl, CFG.TABICL_VERSION)
assert torch.cuda.is_available(), "Enable Kaggle GPU accelerator; V9 foundation inference is not allowed to silently run on CPU"
assert torch.cuda.device_count() >= 1

gpu_info = []
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    gpu_info.append({
        "index": i,
        "name": torch.cuda.get_device_name(i),
        "total_memory_gib": float(props.total_memory / (1024**3)),
    })
print("GPU inventory:", gpu_info)

HF_CACHE = CFG.OUTPUT_DIR / "hf_cache"
HF_CACHE.mkdir(parents=True, exist_ok=True)
try:
    TABICL_CHECKPOINT_PATH = Path(hf_hub_download(
        repo_id=CFG.TABICL_HF_REPO,
        filename=CFG.TABICL_REG_CHECKPOINT,
        cache_dir=str(HF_CACHE),
    ))
except Exception as e:
    raise RuntimeError(
        "Official TabICLv2 checkpoint download failed. Enable Kaggle Internet or attach the exact cached checkpoint."
    ) from e

checkpoint_sha256 = sha256_file(TABICL_CHECKPOINT_PATH)
assert checkpoint_sha256 == CFG.TABICL_REG_SHA256, (
    "Checkpoint hash mismatch; refusing to run an unpinned model", checkpoint_sha256
)

FOUNDATION_MANIFEST = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_available": bool(torch.cuda.is_available()),
    "gpu_count": int(torch.cuda.device_count()),
    "gpus": gpu_info,
    "tabicl_version": installed_tabicl,
    "tabicl_repo": CFG.TABICL_HF_REPO,
    "tabicl_checkpoint": CFG.TABICL_REG_CHECKPOINT,
    "tabicl_checkpoint_path": str(TABICL_CHECKPOINT_PATH),
    "tabicl_checkpoint_sha256": checkpoint_sha256,
    "tabicl_license": "BSD-3-Clause",
    "n_estimators_runtime": CFG.TABICL_RUNTIME_N_ESTIMATORS,
    "n_estimators_screen": CFG.TABICL_SCREEN_N_ESTIMATORS,
    "n_estimators_full": CFG.TABICL_FULL_N_ESTIMATORS,
    "batch_size": CFG.TABICL_BATCH_SIZE,
    "kv_cache": CFG.TABICL_KV_CACHE,
    "offload_mode": CFG.TABICL_OFFLOAD_MODE,
}
with open(CFG.OUTPUT_DIR / "foundation_manifest.json", "w") as f:
    json.dump(FOUNDATION_MANIFEST, f, indent=2)
print(json.dumps(FOUNDATION_MANIFEST, indent=2))


Installing: /usr/bin/python3 -m pip install -q --disable-pip-version-check tabicl==2.2.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.4/310.4 kB 16.3 MB/s eta 0:00:00
GPU inventory: [{'index': 0, 'name': 'Tesla T4', 'total_memory_gib': 14.56219482421875}, {'index': 1, 'name': 'Tesla T4', 'total_memory_gib': 14.56219482421875}]


tabicl-regressor-v2-20260212.ckpt:   0%|          | 0.00/114M [00:00<?, ?B/s]

{
  "python": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "platform": "Linux-6.12.90+-x86_64-with-glibc2.35",
  "torch": "2.10.0+cu128",
  "cuda_available": true,
  "gpu_count": 2,
  "gpus": [
    {
      "index": 0,
      "name": "Tesla T4",
      "total_memory_gib": 14.56219482421875
    },
    {
      "index": 1,
      "name": "Tesla T4",
      "total_memory_gib": 14.56219482421875
    }
  ],
  "tabicl_version": "2.2.0",
  "tabicl_repo": "jingang/TabICL",
  "tabicl_checkpoint": "tabicl-regressor-v2-20260212.ckpt",
  "tabicl_checkpoint_path": "/kaggle/working/oqdq5_v9/hf_cache/models--jingang--TabICL/snapshots/4dcd344ece2c00be9e831fdd35bed57b5ad83e19/tabicl-regressor-v2-20260212.ckpt",
  "tabicl_checkpoint_sha256": "0db9cb538f114e79026bf08f45f41ad8dd7ad2de2aaca9a5ca8cd3bd9748ae7a",
  "tabicl_license": "BSD-3-Clause",
  "n_estimators_runtime": 1,
  "n_estimators_screen": 2,
  "n_estimators_full": 4,
  "batch_size": 1,
  "kv_cache": false,
  "offload_mode": "auto"
}


## 4. V9 unique-respondent data contract and fold-local foundation representation

No supervised global feature selection is used. The full competition feature set is retained except identifiers. Only conservative continuous special-response separation, one global missing-count feature, fold-selected missing flags, and fold-local low-cardinality nominal typing are applied.


In [18]:
# V9 model rows: one row per respondent, preserving first appearance order.
train_uid = train.drop_duplicates(CFG.ID_COL, keep="first").reset_index(drop=True).copy()
test_uid = test.drop_duplicates(CFG.ID_COL, keep="first").reset_index(drop=True).copy()

assert len(train_uid) == train[CFG.ID_COL].nunique()
assert len(test_uid) == test[CFG.ID_COL].nunique() == CFG.EXPECTED_UNIQUE_TEST_IDS
assert train_uid[CFG.ID_COL].tolist() == v2_uid_oof[CFG.ID_COL].tolist()
assert test_uid[CFG.ID_COL].tolist() == v2_uid_test[CFG.ID_COL].tolist()

# Frozen primary fold from V2, now exactly one fold per unique respondent.
uid_fold_map = v2_uid_oof.set_index(CFG.ID_COL)["fold"]
train_uid["__primary_fold"] = train_uid[CFG.ID_COL].map(uid_fold_map).astype(int)
assert train_uid["__primary_fold"].notna().all()

V9_BASE_FEATURES = [c for c in test.columns if c not in {CFG.TARGET, CFG.ID_COL, CFG.HOUSEHOLD_COL}]
V9_CONTINUOUS_SPECIAL = {c: set(v) for c, v in SPECIAL_CODES.items() if c in V9_BASE_FEATURES}


def make_foundation_numeric(df):
    X = df[V9_BASE_FEATURES].copy()
    for c in V9_BASE_FEATURES:
        X[c] = pd.to_numeric(X[c], errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan)
    X["row_missing_count_raw"] = X[V9_BASE_FEATURES].isna().sum(axis=1).astype(np.float32)
    for c, codes in V9_CONTINUOUS_SPECIAL.items():
        flag = f"{c}__special_response"
        X[flag] = X[c].isin(codes).astype(np.float32)
        X.loc[X[c].isin(codes), c] = np.nan
    return X.astype(np.float32)

V9_NUM_TRAIN = make_foundation_numeric(train_uid)
V9_NUM_TEST = make_foundation_numeric(test_uid)


def select_missing_flags(Xtr, ytr, topk=16):
    rows = []
    for c in V9_BASE_FEATURES:
        if c not in Xtr:
            continue
        m = Xtr[c].isna().to_numpy()
        rate = float(m.mean())
        if rate < 0.02 or rate > 0.98 or m.sum() < 100 or (~m).sum() < 100:
            continue
        diff = abs(float(np.mean(ytr[m])) - float(np.mean(ytr[~m])))
        rows.append((diff, c, rate))
    rows = sorted(rows, reverse=True)
    return [c for _, c, _ in rows[:topk]], rows


def append_missing_flags(A, B, C, cols):
    A, B, C = A.copy(), B.copy(), C.copy()
    for c in cols:
        for frame in (A, B, C):
            frame[f"{c}__missing_flag"] = frame[c].isna().astype(np.float32)
    return A, B, C


def infer_tabicl_nominal_cols(Xtr, ytr, max_unique=15, rho_cut=0.55, max_cols=60):
    rows = []
    for c in V9_BASE_FEATURES:
        if c not in Xtr:
            continue
        s = Xtr[c]
        nu = int(s.nunique(dropna=True))
        if nu < 3 or nu > max_unique:
            continue
        z = pd.DataFrame({"x": s, "y": ytr}).dropna()
        if len(z) < 300:
            continue
        g = z.groupby("x").y.agg(["mean", "size"]).reset_index()
        g = g[g["size"] >= 30]
        if len(g) < 3:
            continue
        rho = spearmanr(g["x"].astype(float), g["mean"].astype(float)).correlation
        rho = 0.0 if not np.isfinite(rho) else float(rho)
        score = (1 - abs(rho)) + 0.02 * (max_unique - nu)
        if abs(rho) < rho_cut:
            rows.append((score, c, nu, rho))
    rows = sorted(rows, reverse=True)[:max_cols]
    return [r[1] for r in rows], rows


def make_tabicl_mixed(A, B, C, nominal_cols):
    A, B, C = A.copy(), B.copy(), C.copy()
    for c in nominal_cols:
        if c not in A:
            continue
        train_levels = [str(v) for v in A[c].dropna().unique().tolist()]
        level_set = set(train_levels)
        categories = train_levels + ["__UNK__"]
        def cvt(s):
            vals = s.map(lambda v: np.nan if pd.isna(v) else (str(v) if str(v) in level_set else "__UNK__"))
            return pd.Categorical(vals, categories=categories)
        A[c], B[c], C[c] = cvt(A[c]), cvt(B[c]), cvt(C[c])
    return A, B, C


def select_resource_topk(A, ytr, group_ids, fold, topk=96):
    # Resource fallback only: supervised selection is entirely inside the outer-train fold.
    ytr = np.asarray(ytr)
    strata_inner = pd.Series(ytr).astype(str).to_numpy()
    cv = StratifiedGroupKFold(n_splits=CFG.INNER_N_FOLDS, shuffle=True, random_state=9700 + int(fold))
    i_tr, i_va = next(cv.split(A, strata_inner, groups=np.asarray(group_ids)))
    model = None
    for gpu_flag in ([True, False] if CFG.USE_GPU else [False]):
        try:
            model = make_lgb_fs(9700 + int(fold), use_gpu=gpu_flag)
            model.fit(
                A.iloc[i_tr], ytr[i_tr],
                eval_set=[(A.iloc[i_va], ytr[i_va])],
                callbacks=[lgb.early_stopping(CFG.INNER_FS_EARLY_STOPPING, verbose=False)],
            )
            break
        except Exception as e:
            if gpu_flag:
                print("Resource selector LightGBM GPU -> CPU fallback:", repr(e))
                gc.collect()
                continue
            raise
    gain = model.booster_.feature_importance(importance_type="gain")
    ranked = [f for _, f in sorted(zip(gain, A.columns.tolist()), reverse=True)]
    selected = ranked[: min(int(topk), len(ranked))]
    report = pd.DataFrame({"feature": A.columns, "gain": gain}).sort_values("gain", ascending=False)
    del model
    gc.collect()
    return selected, report


def build_v9_fold_views(fold, feature_mode="full"):
    fold_ids = train_uid["__primary_fold"].to_numpy()
    tr = np.where(fold_ids != fold)[0]
    va = np.where(fold_ids == fold)[0]
    y_all = train_uid[CFG.TARGET].astype(int).to_numpy()

    A = V9_NUM_TRAIN.iloc[tr].copy()
    B = V9_NUM_TRAIN.iloc[va].copy()
    C = V9_NUM_TEST.copy()

    flags, flag_report = select_missing_flags(A, y_all[tr], CFG.TABICL_MISSING_FLAG_TOPK)
    A, B, C = append_missing_flags(A, B, C, flags)

    resource_selected = None
    resource_report = None
    if feature_mode == "top96":
        resource_selected, resource_report = select_resource_topk(
            A, y_all[tr], train_uid.iloc[tr][CFG.ID_COL].astype(str).to_numpy(), fold, CFG.TABICL_RESOURCE_TOPK
        )
        A, B, C = A[resource_selected].copy(), B[resource_selected].copy(), C[resource_selected].copy()
        assert A.shape[1] <= CFG.TABICL_RESOURCE_TOPK
    elif feature_mode != "full":
        raise ValueError(f"Unknown V9 feature_mode={feature_mode}")

    nominal, nominal_report = infer_tabicl_nominal_cols(A, y_all[tr])
    A, B, C = make_tabicl_mixed(A, B, C, nominal)

    assert A.columns.tolist() == B.columns.tolist() == C.columns.tolist()
    assert CFG.ID_COL not in A.columns and CFG.HOUSEHOLD_COL not in A.columns and CFG.TARGET not in A.columns
    return tr, va, A, B, C, flags, flag_report, nominal, nominal_report, resource_selected, resource_report

print("V9 unique train/test:", train_uid.shape, test_uid.shape)
print("V9 base feature width:", len(V9_BASE_FEATURES), "numeric width before fold flags:", V9_NUM_TRAIN.shape[1])


V9 unique train/test: (94663, 180) (23666, 178)
V9 base feature width: 176 numeric width before fold flags: 183


## 5. TabICL offloading/runtime helpers

TabICL inference exposes one device per estimator, not model-parallel multi-GPU inference. To avoid an invalid custom model-parallel path, V9 routes independent folds across `cuda:0` and `cuda:1` when two T4s exist, but each estimator remains a supported single-device TabICL run.


In [19]:
def choose_offload_root():
    candidates = [Path("/kaggle/temp"), Path("/kaggle/working"), CFG.OUTPUT_DIR]
    rows = []
    for p in candidates:
        try:
            p.mkdir(parents=True, exist_ok=True)
            du = shutil.disk_usage(p)
            rows.append((du.free, p))
        except Exception:
            continue
    if not rows:
        raise RuntimeError("No writable directory available for TabICL disk offloading")
    free, root = max(rows, key=lambda x: x[0])
    out = root / "oqdq5_v9_tabicl_offload"
    out.mkdir(parents=True, exist_ok=True)
    print(f"Offload root: {out} | free={free/(1024**3):.2f} GiB")
    return out

OFFLOAD_ROOT = choose_offload_root()


def cleanup_cuda_and_offload(path=None):
    gc.collect()
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            try:
                with torch.cuda.device(i):
                    torch.cuda.empty_cache()
            except Exception:
                pass
    if path is not None:
        try:
            shutil.rmtree(path, ignore_errors=True)
        except Exception:
            pass


def fold_device(fold):
    # Correct use of T4 x2 without unsupported model parallelism.
    if torch.cuda.device_count() >= 2:
        return f"cuda:{int(fold) % 2}"
    return "cuda:0"


class TabICLRunError(RuntimeError):
    def __init__(self, message, failure):
        super().__init__(message)
        self.failure = failure


def run_tabicl_fold(fold, n_estimators, predict_test, stage, verbose=False, feature_mode="full"):
    tr, va, A, B, C, flags, flag_report, nominal, nominal_report, resource_selected, resource_report = build_v9_fold_views(fold, feature_mode=feature_mode)
    device = fold_device(fold)
    fold_dir = OFFLOAD_ROOT / f"{stage}_fold{fold}_seed{9400 + fold}"
    cleanup_cuda_and_offload(fold_dir)
    fold_dir.mkdir(parents=True, exist_ok=True)

    y_all = train_uid[CFG.TARGET].astype(float).to_numpy()
    seed = 9400 + fold + 100 * int(n_estimators)
    started = time.time()
    model = None
    try:
        model = TabICLRegressor(
            n_estimators=int(n_estimators),
            batch_size=CFG.TABICL_BATCH_SIZE,
            kv_cache=CFG.TABICL_KV_CACHE,
            model_path=str(TABICL_CHECKPOINT_PATH),
            allow_auto_download=False,
            device=device,
            use_amp=CFG.TABICL_USE_AMP,
            use_fa3=CFG.TABICL_USE_FA3,
            offload_mode=CFG.TABICL_OFFLOAD_MODE,
            disk_offload_dir=str(fold_dir),
            random_state=seed,
            verbose=bool(verbose),
        )
        model.fit(A, y_all[tr])
        pva = np.asarray(model.predict(B), dtype=np.float64).reshape(-1)
        pte = np.asarray(model.predict(C), dtype=np.float64).reshape(-1) if predict_test else None
        assert len(pva) == len(va) and np.isfinite(pva).all()
        if predict_test:
            assert pte is not None and len(pte) == len(test_uid) and np.isfinite(pte).all()
        runtime = time.time() - started
        info = {
            "stage": stage,
            "fold": int(fold),
            "device": device,
            "n_estimators": int(n_estimators),
            "feature_mode": feature_mode,
            "train_rows": int(len(tr)),
            "valid_rows": int(len(va)),
            "test_rows": int(len(test_uid)) if predict_test else 0,
            "feature_count": int(A.shape[1]),
            "runtime_sec": float(runtime),
            "missing_flags": flags,
            "nominal_cols": nominal,
            "flag_report_top": [list(x) for x in flag_report[:CFG.TABICL_MISSING_FLAG_TOPK]],
            "nominal_report": [list(x) for x in nominal_report],
            "resource_selected_features": resource_selected,
            "resource_selector_top": (resource_report.head(CFG.TABICL_RESOURCE_TOPK).to_dict("records") if resource_report is not None else None),
            "offload_dir": str(fold_dir),
            "error": None,
        }
        return va, pva, pte, info
    except Exception as e:
        failure = {
            "stage": stage,
            "fold": int(fold),
            "device": device,
            "n_estimators": int(n_estimators),
            "feature_mode": feature_mode,
            "runtime_sec_before_failure": float(time.time() - started),
            "offload_dir": str(fold_dir),
            "error": repr(e),
        }
        with open(CFG.OUTPUT_DIR / f"FAILURE_{stage}_fold{fold}.json", "w") as f:
            json.dump(failure, f, indent=2)
        raise TabICLRunError(f"TabICL {stage} fold {fold} failed; no V2 submission fallback is permitted.", failure) from e
    finally:
        if model is not None:
            del model
        cleanup_cuda_and_offload(fold_dir)


Offload root: /kaggle/temp/oqdq5_v9_tabicl_offload | free=1026.77 GiB


## 6. Stage R — full-context runtime proof

This intentionally runs one full validation fold with the full outer-training context, all V9 features, `n_estimators=1`, `batch_size=1`, and explicit disk offloading. A runtime failure stops the notebook.


In [20]:
def is_resource_failure_text(text):
    t = str(text).lower()
    resource_patterns = [
        "cuda out of memory", "out of memory", "memoryerror", "cannot allocate memory",
        "not enough memory", "insufficient memory", "no space left on device",
    ]
    if any(p in t for p in resource_patterns):
        return True
    return ("requires" in t and (" gib" in t or " gb" in t) and "memory" in t)

RUNTIME_FOLD = 0
V9_FEATURE_MODE = "full"
RESOURCE_FALLBACK = {"used": False, "reason": None, "from": "full", "to": None}
try:
    va_r, pred_r, _, runtime_info = run_tabicl_fold(
        fold=RUNTIME_FOLD,
        n_estimators=CFG.TABICL_RUNTIME_N_ESTIMATORS,
        predict_test=False,
        stage="runtime_proof_full",
        verbose=True,
        feature_mode="full",
    )
except TabICLRunError as e:
    original_error = e.failure.get("error", repr(e))
    if not is_resource_failure_text(original_error):
        raise
    print("Full-feature Stage R hit a resource limit. Activating the preregistered fold-safe top-96 runtime fallback.")
    V9_FEATURE_MODE = "top96"
    RESOURCE_FALLBACK = {"used": True, "reason": original_error, "from": "full", "to": "top96"}
    va_r, pred_r, _, runtime_info = run_tabicl_fold(
        fold=RUNTIME_FOLD,
        n_estimators=CFG.TABICL_RUNTIME_N_ESTIMATORS,
        predict_test=False,
        stage="runtime_proof_top96",
        verbose=True,
        feature_mode="top96",
    )

assert len(pred_r) == int((train_uid["__primary_fold"] == RUNTIME_FOLD).sum())
runtime_info["resource_fallback"] = RESOURCE_FALLBACK
with open(CFG.OUTPUT_DIR / "runtime_proof.json", "w") as f:
    json.dump(runtime_info, f, indent=2)
with open(CFG.OUTPUT_DIR / "resource_fallback.json", "w") as f:
    json.dump(RESOURCE_FALLBACK, f, indent=2)
print(json.dumps(runtime_info, indent=2))
print("Stage R passed with feature mode:", V9_FEATURE_MODE)
del pred_r
gc.collect()


Columns classified as categorical: ['works_in_healthcare', 'has_disability_income', 'difficulty_social_activities', 'has_lowincome_gov_insurance', 'has_wage_income', 'ever_smoked_cigar', 'has_indigenous_insurance', 'takes_depression_medication', 'virtual_medical_appointment', 'lung_disease_ever', 'ever_served_military', 'food_security_4cat', 'has_health_insurance', 'hospitalized_overnight', 'heart_attack_ever', 'depression_ever', 'worry_about_medical_costs', 'takes_anxiety_medication', 'health_insurance_type', 'sexual_orientation', 'food_security_3cat', 'has_state_health_plan', 'could_not_afford_balanced_meals', 'insurance_paid_gov_lowincome_plan1', 'anxiety_frequency', 'took_prescription_medication', 'received_food_assistance', 'weakened_immune_medication', 'bmi_category', 'heart_disease_ever', 'trouble_paying_medical_bills', 'takes_mental_health_medication', 'marital_status_detailed', 'delayed_therapy_due_to_cost', 'insurance_has_deductible_plan1', 'home_ownership', 'anxiety_ever', '

Processing tf_col:   0%|          | 0/9 [00:00<?, ?batch/s]


Available GPU memory: 14.15GB, seq_len: 199, estimated batch size for tf_row: 9820
Offload decision: mode=GPU (reason=user_gpu_fits: 185MB <= 14487MB gpu free)
Output size: 0.18GB (repeat=1), CPU free: 4.40GB, GPU free: 14.15GB, Disk free (effective): 0.00GB @ None


Processing tf_row:   0%|          | 0/10 [00:00<?, ?batch/s]


Available GPU memory: 13.97GB, seq_len: 94663, estimated batch size for tf_icl: 6
Offload decision: mode=GPU (reason=user_gpu_fits: 361MB <= 14301MB gpu free)
Output size: 0.35GB (repeat=1), CPU free: 4.36GB, GPU free: 13.97GB, Disk free (effective): 0.00GB @ None
{
  "stage": "runtime_proof_full",
  "fold": 0,
  "device": "cuda:0",
  "n_estimators": 1,
  "feature_mode": "full",
  "train_rows": 75730,
  "valid_rows": 18933,
  "test_rows": 0,
  "feature_count": 199,
  "runtime_sec": 53.58839201927185,
  "missing_flags": [
    "ever_married",
    "anxiety_level",
    "difficulty_walking_medium",
    "difficulty_walking_short",
    "difficulty_climbing_steps",
    "takes_mental_health_medication",
    "insurance_marketplace_plan1",
    "insurance_paid_other_plan1",
    "insurance_paid_gov_elderly_plan1",
    "insurance_covers_vision_plan1",
    "insurance_paid_gov_lowincome_plan1",
    "insurance_paid_self_plan1",
    "insurance_has_deductible_plan1",
    "insurance_covers_rx_plan1",
   

99

## 7. Stage S — preregistered two-fold scientific screen

Folds `0` and `2` are evaluated with `n_estimators=2`. QWK screening uses cross-screen thresholds: each fold's thresholds are learned only from the *other* screen fold. This keeps the held-out fold untouched even during the cheap screen.


In [21]:
def boundary_diagnostics(y_true, score):
    y_true = np.asarray(y_true, int)
    score = np.asarray(score, float)
    m12 = np.isin(y_true, [1, 2])
    return {
        "auc_gt1": float(roc_auc_score((y_true > 1).astype(int), score)),
        "auc_1v2": float(roc_auc_score((y_true[m12] == 2).astype(int), score[m12])),
        "auc_gt2": float(roc_auc_score((y_true > 2).astype(int), score)),
        "auc_gt3": float(roc_auc_score((y_true > 3).astype(int), score)),
        "rmse": float(mean_squared_error(y_true, score) ** 0.5),
        "mae": float(mean_absolute_error(y_true, score)),
        "spearman": float(spearmanr(y_true, score).correlation),
    }

screen_pred = {}
screen_infos = []
for f in CFG.TABICL_SCREEN_FOLDS:
    va, pva, _, info = run_tabicl_fold(
        fold=f,
        n_estimators=CFG.TABICL_SCREEN_N_ESTIMATORS,
        predict_test=False,
        stage="screen",
        verbose=False,
        feature_mode=V9_FEATURE_MODE,
    )
    assert np.array_equal(np.where(train_uid["__primary_fold"].to_numpy() == f)[0], va)
    screen_pred[int(f)] = pva
    screen_infos.append(info)
    np.savez_compressed(CFG.OUTPUT_DIR / f"stage_s_fold{f}_predictions.npz", valid_index=va, valid_prediction=pva)

# Cross-screen thresholding: threshold for each fold comes from the other screen fold only.
screen_rows = []
for f in CFG.TABICL_SCREEN_FOLDS:
    other = [x for x in CFG.TABICL_SCREEN_FOLDS if x != f][0]
    m = train_uid["__primary_fold"].to_numpy() == f
    mo = train_uid["__primary_fold"].to_numpy() == other
    y_f = train_uid.loc[m, CFG.TARGET].astype(int).to_numpy()
    y_o = train_uid.loc[mo, CFG.TARGET].astype(int).to_numpy()
    tab_f = screen_pred[f]
    tab_o = screen_pred[other]
    v2_f = v2_uid_oof.loc[m, "v2_score"].to_numpy(float)
    v2_o = v2_uid_oof.loc[mo, "v2_score"].to_numpy(float)

    tab_t, _ = optimize_thresholds(y_o, tab_o, maxiter=130)
    v2_t, _ = optimize_thresholds(y_o, v2_o, maxiter=130)
    tab_q = qwk(y_f, apply_thresholds(tab_f, tab_t))
    v2_q = qwk(y_f, apply_thresholds(v2_f, v2_t))

    td = boundary_diagnostics(y_f, tab_f)
    vd = boundary_diagnostics(y_f, v2_f)
    rec = {
        "fold": int(f),
        **{f"tabicl_{k}": v for k, v in td.items()},
        **{f"v2_{k}": v for k, v in vd.items()},
        "delta_auc_1v2": td["auc_1v2"] - vd["auc_1v2"],
        "delta_auc_gt1": td["auc_gt1"] - vd["auc_gt1"],
        "crossscreen_tabicl_qwk": float(tab_q),
        "crossscreen_v2_qwk": float(v2_q),
        "crossscreen_qwk_gain": float(tab_q - v2_q),
        "residual_corr_vs_v2": float(np.corrcoef(y_f - tab_f, y_f - v2_f)[0, 1]),
        "threshold_source_fold": int(other),
    }
    screen_rows.append(rec)

screen_df = pd.DataFrame(screen_rows).sort_values("fold")
screen_df.to_csv(CFG.OUTPUT_DIR / "stage_s_metrics.csv", index=False)
with open(CFG.OUTPUT_DIR / "stage_s_runtime_and_preprocessing.json", "w") as f:
    json.dump(screen_infos, f, indent=2)
display(screen_df)

mean_delta_auc_1v2 = float(screen_df.delta_auc_1v2.mean())
mean_delta_auc_gt1 = float(screen_df.delta_auc_gt1.mean())
mean_qwk_gain = float(screen_df.crossscreen_qwk_gain.mean())
mean_resid_corr = float(screen_df.residual_corr_vs_v2.mean())
no_catastrophic = bool(
    (screen_df.delta_auc_1v2 >= CFG.SCREEN_MIN_PER_FOLD_DELTA_AUC).all()
    and (screen_df.delta_auc_gt1 >= CFG.SCREEN_MIN_PER_FOLD_DELTA_AUC).all()
)
ranking_gate = bool(no_catastrophic and (
    mean_delta_auc_1v2 >= CFG.SCREEN_MIN_MEAN_DELTA_AUC
    or mean_delta_auc_gt1 >= CFG.SCREEN_MIN_MEAN_DELTA_AUC
))
qwk_gate = bool(mean_qwk_gain >= CFG.SCREEN_MIN_MEAN_DELTA_QWK)
diversity_gate = bool(
    mean_qwk_gain >= CFG.SCREEN_DIVERSITY_MIN_MEAN_QWK_GAIN
    and mean_resid_corr < CFG.SCREEN_MAX_RESIDUAL_CORR_DIVERSITY
)
V9_PROMOTED = bool(ranking_gate or qwk_gate or diversity_gate)

STAGE_S_SUMMARY = {
    "feature_mode": V9_FEATURE_MODE,
    "resource_fallback": RESOURCE_FALLBACK,
    "mean_delta_auc_1v2": mean_delta_auc_1v2,
    "mean_delta_auc_gt1": mean_delta_auc_gt1,
    "mean_crossscreen_qwk_gain": mean_qwk_gain,
    "mean_residual_corr_vs_v2": mean_resid_corr,
    "no_catastrophic_auc_regression": no_catastrophic,
    "ranking_gate": ranking_gate,
    "qwk_gate": qwk_gate,
    "diversity_gate": diversity_gate,
    "promoted_to_full_cv": V9_PROMOTED,
}
with open(CFG.OUTPUT_DIR / "stage_s_summary.json", "w") as f:
    json.dump(STAGE_S_SUMMARY, f, indent=2)
print(json.dumps(STAGE_S_SUMMARY, indent=2))

if not V9_PROMOTED:
    print("V9 FALSIFIED AT STAGE S. Full 5-fold/test inference and submission generation are intentionally skipped.")


,fold,tabicl_auc_gt1,tabicl_auc_1v2,tabicl_auc_gt2,tabicl_auc_gt3,tabicl_rmse,tabicl_mae,tabicl_spearman,v2_auc_gt1,v2_auc_1v2,v2_auc_gt2,v2_auc_gt3,v2_rmse,v2_mae,v2_spearman,delta_auc_1v2,delta_auc_gt1,crossscreen_tabicl_qwk,crossscreen_v2_qwk,crossscreen_qwk_gain,residual_corr_vs_v2,threshold_source_fold
0,0,0.773954,0.758791,0.879820,0.834960,0.518079,0.430856,0.502044,0.781769,0.767062,0.881479,0.832184,0.512644,0.426232,0.514814,-0.008271,-0.007815,0.479360,0.487756,-0.008397,0.988559,2
1,2,0.767603,0.753291,0.867079,0.795182,0.525878,0.432489,0.488656,0.776690,0.762839,0.871740,0.796262,0.519830,0.428012,0.503966,-0.009548,-0.009086,0.463204,0.473889,-0.010685,0.988276,0


{
  "feature_mode": "full",
  "resource_fallback": {
    "used": false,
    "reason": null,
    "from": "full",
    "to": null
  },
  "mean_delta_auc_1v2": -0.008909788847464906,
  "mean_delta_auc_gt1": -0.008450922316947773,
  "mean_crossscreen_qwk_gain": -0.009541022876559113,
  "mean_residual_corr_vs_v2": 0.988417058532844,
  "no_catastrophic_auc_regression": false,
  "ranking_gate": false,
  "qwk_gate": false,
  "diversity_gate": false,
  "promoted_to_full_cv": false
}
V9 FALSIFIED AT STAGE S. Full 5-fold/test inference and submission generation are intentionally skipped.


## 8. Full five-fold TabICLv2 OOF + test inference, only if Stage S survives

The full run uses `n_estimators=4`. Every validation respondent receives exactly one TabICL score. Test prediction is the mean of the five outer-fold contexts.


In [22]:
V9_FULL_AVAILABLE = False
v9_full_oof = np.full(len(train_uid), np.nan, dtype=np.float64)
v9_test_fold_scores = {}
v9_full_infos = []

if V9_PROMOTED:
    for f in sorted(train_uid["__primary_fold"].unique().tolist()):
        print(f"\n[FULL TabICLv2] fold={f} device={fold_device(f)}")
        va, pva, pte, info = run_tabicl_fold(
            fold=int(f),
            n_estimators=CFG.TABICL_FULL_N_ESTIMATORS,
            predict_test=True,
            stage="full",
            verbose=False,
            feature_mode=V9_FEATURE_MODE,
        )
        v9_full_oof[va] = pva
        v9_test_fold_scores[int(f)] = pte
        v9_full_infos.append(info)
        np.savez_compressed(CFG.OUTPUT_DIR / f"full_fold{f}_predictions.npz", valid_index=va, valid_prediction=pva, test_prediction=pte)

    assert np.isfinite(v9_full_oof).all()
    assert len(v9_test_fold_scores) == CFG.N_FOLDS
    v9_test_score = np.mean(np.column_stack([v9_test_fold_scores[f] for f in sorted(v9_test_fold_scores)]), axis=1)
    assert len(v9_test_score) == len(test_uid) and np.isfinite(v9_test_score).all()
    V9_FULL_AVAILABLE = True

    with open(CFG.OUTPUT_DIR / "full_runtime_and_preprocessing.json", "w") as f:
        json.dump(v9_full_infos, f, indent=2)
else:
    print("Full CV skipped by preregistered Stage-S falsification gate.")


Full CV skipped by preregistered Stage-S falsification gate.


## 9. Evaluator-aligned full validation, paired bootstrap, and nested blend

All headline QWK values are unique-respondent and cross-fitted. Global threshold optimization is never used as the scientific headline.


In [23]:
def crossfit_labels_and_thresholds(y_true, score, folds, maxiter=150):
    y_true = np.asarray(y_true, int)
    score = np.asarray(score, float)
    folds = np.asarray(folds, int)
    labels = np.zeros(len(y_true), dtype=int)
    rows = []
    for f in sorted(np.unique(folds)):
        tr = folds != f
        va = folds == f
        t, train_q = optimize_thresholds(y_true[tr], score[tr], maxiter=maxiter)
        labels[va] = apply_thresholds(score[va], t)
        rows.append({
            "fold": int(f), "t1": float(t[0]), "t2": float(t[1]), "t3": float(t[2]),
            "meta_train_qwk": float(train_q),
            "heldout_qwk": float(qwk(y_true[va], labels[va])),
        })
    return labels, pd.DataFrame(rows)


def weighted_qwk_from_pairs(y_true, y_pred, sample_idx):
    # respondent-level bootstrap; sampling is with replacement.
    return qwk(np.asarray(y_true)[sample_idx], np.asarray(y_pred)[sample_idx])


def paired_bootstrap_delta(y_true, pred_a, pred_b, reps=1000, seed=20260905):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    deltas = np.empty(reps, dtype=np.float32)
    for i in range(reps):
        idx = rng.integers(0, n, size=n)
        deltas[i] = weighted_qwk_from_pairs(y_true, pred_a, idx) - weighted_qwk_from_pairs(y_true, pred_b, idx)
    return {
        "reps": int(reps),
        "median_delta": float(np.median(deltas)),
        "ci95_low": float(np.quantile(deltas, 0.025)),
        "ci95_high": float(np.quantile(deltas, 0.975)),
        "p_delta_gt_0": float(np.mean(deltas > 0)),
    }, deltas


def class_distribution(labels):
    s = pd.Series(np.asarray(labels, int)).value_counts(normalize=True).reindex([1,2,3,4], fill_value=0.0)
    return s.to_numpy(float)


def total_variation(p, q):
    return float(0.5 * np.abs(np.asarray(p) - np.asarray(q)).sum())

FULL_SUMMARY = {"full_available": bool(V9_FULL_AVAILABLE), "feature_mode": V9_FEATURE_MODE, "resource_fallback": RESOURCE_FALLBACK}
RECOMMENDED_CANDIDATE = None

if V9_FULL_AVAILABLE:
    y_u = train_uid[CFG.TARGET].astype(int).to_numpy()
    folds_u = train_uid["__primary_fold"].astype(int).to_numpy()
    v2_u = v2_uid_oof["v2_score"].to_numpy(float)

    # Standalone honest cross-fitted labels.
    tab_labels, tab_thr_df = crossfit_labels_and_thresholds(y_u, v9_full_oof, folds_u)
    v2_labels, v2_thr_df = crossfit_labels_and_thresholds(y_u, v2_u, folds_u)
    tab_cf = float(qwk(y_u, tab_labels))
    v2_cf = float(qwk(y_u, v2_labels))
    tab_gain = float(tab_cf - v2_cf)

    tab_diag = boundary_diagnostics(y_u, v9_full_oof)
    v2_diag = boundary_diagnostics(y_u, v2_u)
    residual_corr = float(np.corrcoef(y_u - v9_full_oof, y_u - v2_u)[0,1])

    fold_metrics = []
    for f in sorted(np.unique(folds_u)):
        m = folds_u == f
        tab_f_q = qwk(y_u[m], tab_labels[m])
        v2_f_q = qwk(y_u[m], v2_labels[m])
        td = boundary_diagnostics(y_u[m], v9_full_oof[m])
        vd = boundary_diagnostics(y_u[m], v2_u[m])
        fold_metrics.append({
            "fold": int(f), "tabicl_qwk": float(tab_f_q), "v2_qwk": float(v2_f_q),
            "qwk_gain": float(tab_f_q - v2_f_q),
            "tabicl_auc_1v2": td["auc_1v2"], "v2_auc_1v2": vd["auc_1v2"],
            "delta_auc_1v2": td["auc_1v2"] - vd["auc_1v2"],
            "tabicl_auc_gt1": td["auc_gt1"], "v2_auc_gt1": vd["auc_gt1"],
            "delta_auc_gt1": td["auc_gt1"] - vd["auc_gt1"],
        })
    fold_metrics_df = pd.DataFrame(fold_metrics)

    tab_boot, tab_boot_deltas = paired_bootstrap_delta(y_u, tab_labels, v2_labels, CFG.BOOTSTRAP_REPS, seed=20260905)

    # Preregistered nested blend: alpha and thresholds are selected only on the other four folds.
    blend_score_oof = np.full(len(y_u), np.nan, dtype=float)
    blend_labels = np.zeros(len(y_u), dtype=int)
    blend_selection_rows = []
    for f in sorted(np.unique(folds_u)):
        tr = folds_u != f
        va = folds_u == f
        best = None
        for alpha in CFG.BLEND_ALPHA_GRID:
            s_tr = (1.0 - alpha) * v2_u[tr] + alpha * v9_full_oof[tr]
            t, train_q = optimize_thresholds(y_u[tr], s_tr, maxiter=150)
            rec = {"alpha": float(alpha), "thresholds": t, "meta_train_qwk": float(train_q)}
            if best is None or rec["meta_train_qwk"] > best["meta_train_qwk"] + 1e-12 or (
                abs(rec["meta_train_qwk"] - best["meta_train_qwk"]) <= 1e-12 and rec["alpha"] < best["alpha"]
            ):
                best = rec
        alpha = best["alpha"]
        t = np.asarray(best["thresholds"], float)
        s_va = (1.0 - alpha) * v2_u[va] + alpha * v9_full_oof[va]
        blend_score_oof[va] = s_va
        blend_labels[va] = apply_thresholds(s_va, t)
        bq = float(qwk(y_u[va], blend_labels[va]))
        vq = float(qwk(y_u[va], v2_labels[va]))
        blend_selection_rows.append({
            "fold": int(f), "alpha": float(alpha), "t1": float(t[0]), "t2": float(t[1]), "t3": float(t[2]),
            "meta_train_qwk": float(best["meta_train_qwk"]), "heldout_blend_qwk": bq,
            "heldout_v2_qwk": vq, "heldout_gain": bq-vq,
        })

    assert np.isfinite(blend_score_oof).all()
    blend_selection_df = pd.DataFrame(blend_selection_rows)
    blend_cf = float(qwk(y_u, blend_labels))
    blend_gain = float(blend_cf - v2_cf)
    blend_boot, blend_boot_deltas = paired_bootstrap_delta(y_u, blend_labels, v2_labels, CFG.BOOTSTRAP_REPS, seed=20260906)
    blend_diag = boundary_diagnostics(y_u, blend_score_oof)

    # Robust test parameters from fold-level meta-selection.
    tab_median_thresholds = tab_thr_df[["t1","t2","t3"]].median(axis=0).to_numpy(float)
    blend_median_alpha = float(blend_selection_df["alpha"].median())
    blend_median_thresholds = blend_selection_df[["t1","t2","t3"]].median(axis=0).to_numpy(float)
    blend_test_score = (1.0 - blend_median_alpha) * v2_uid_test["v2_score"].to_numpy(float) + blend_median_alpha * v9_test_score

    tab_test_class = apply_thresholds(v9_test_score, tab_median_thresholds)
    blend_test_class = apply_thresholds(blend_test_score, blend_median_thresholds)
    tab_oof_dist = class_distribution(tab_labels)
    tab_test_dist = class_distribution(tab_test_class)
    blend_oof_dist = class_distribution(blend_labels)
    blend_test_dist = class_distribution(blend_test_class)
    tab_dist_tv = total_variation(tab_oof_dist, tab_test_dist)
    blend_dist_tv = total_variation(blend_oof_dist, blend_test_dist)

    tab_positive_folds = int((fold_metrics_df.qwk_gain >= 0).sum())
    tab_worst_fold = float(fold_metrics_df.qwk_gain.min())
    blend_positive_folds = int((blend_selection_df.heldout_gain >= 0).sum())
    blend_worst_fold = float(blend_selection_df.heldout_gain.min())

    tab_scientific_positive = bool(
        tab_gain >= CFG.SCIENTIFIC_MIN_QWK_GAIN
        and tab_boot["p_delta_gt_0"] >= CFG.SCIENTIFIC_BOOTSTRAP_P_POSITIVE
    )
    tab_submission_worthy = bool(
        tab_gain >= CFG.SUBMIT_MIN_QWK_GAIN
        and tab_boot["p_delta_gt_0"] >= CFG.SUBMIT_BOOTSTRAP_P_POSITIVE
        and tab_positive_folds >= CFG.SUBMIT_MIN_POSITIVE_FOLDS
        and tab_worst_fold >= CFG.SUBMIT_MIN_WORST_FOLD_GAIN
        and tab_dist_tv <= CFG.MAX_TEST_OOF_CLASS_TV
    )
    blend_scientific_positive = bool(
        blend_gain >= CFG.SCIENTIFIC_MIN_QWK_GAIN
        and blend_boot["p_delta_gt_0"] >= CFG.SCIENTIFIC_BOOTSTRAP_P_POSITIVE
    )
    blend_submission_worthy = bool(
        blend_gain >= CFG.SUBMIT_MIN_QWK_GAIN
        and blend_boot["p_delta_gt_0"] >= CFG.SUBMIT_BOOTSTRAP_P_POSITIVE
        and blend_positive_folds >= CFG.SUBMIT_MIN_POSITIVE_FOLDS
        and blend_worst_fold >= CFG.SUBMIT_MIN_WORST_FOLD_GAIN
        and blend_dist_tv <= CFG.MAX_TEST_OOF_CLASS_TV
    )

    breakthrough = bool(
        max(tab_gain, blend_gain) >= CFG.BREAKTHROUGH_MIN_QWK_GAIN
        or (tab_diag["auc_1v2"] - v2_diag["auc_1v2"]) >= CFG.BREAKTHROUGH_MIN_AUC_GAIN
        or (tab_diag["auc_gt1"] - v2_diag["auc_gt1"]) >= CFG.BREAKTHROUGH_MIN_AUC_GAIN
    )

    # Artifacts.
    oof_out = pd.DataFrame({
        CFG.ID_COL: train_uid[CFG.ID_COL].values,
        CFG.TARGET: y_u,
        "fold": folds_u,
        "v2_score": v2_u,
        "tabicl_score": v9_full_oof,
        "blend_crossfit_score": blend_score_oof,
        "v2_crossfit_class": v2_labels,
        "tabicl_crossfit_class": tab_labels,
        "blend_crossfit_class": blend_labels,
    })
    oof_out.to_csv(CFG.OUTPUT_DIR / "oof_v9.csv", index=False)

    test_out = pd.DataFrame({CFG.ID_COL: test_uid[CFG.ID_COL].values, "v2_score": v2_uid_test["v2_score"].values})
    for f in sorted(v9_test_fold_scores):
        test_out[f"tabicl_fold{f}"] = v9_test_fold_scores[f]
    test_out["tabicl_score"] = v9_test_score
    test_out["blend_score"] = blend_test_score
    test_out.to_csv(CFG.OUTPUT_DIR / "test_predictions_v9.csv", index=False)

    fold_metrics_df.to_csv(CFG.OUTPUT_DIR / "fold_metrics_v9.csv", index=False)
    tab_thr_df.to_csv(CFG.OUTPUT_DIR / "tabicl_fold_thresholds_v9.csv", index=False)
    v2_thr_df.to_csv(CFG.OUTPUT_DIR / "v2_fold_thresholds_unique_v9.csv", index=False)
    blend_selection_df.to_csv(CFG.OUTPUT_DIR / "blend_selection_v9.csv", index=False)
    np.savez_compressed(
        CFG.OUTPUT_DIR / "bootstrap_deltas_v9.npz",
        tabicl_minus_v2=tab_boot_deltas,
        blend_minus_v2=blend_boot_deltas,
    )

    FULL_SUMMARY.update({
        "v2_crossfit_qwk": v2_cf,
        "tabicl_crossfit_qwk": tab_cf,
        "tabicl_qwk_gain": tab_gain,
        "blend_crossfit_qwk": blend_cf,
        "blend_qwk_gain": blend_gain,
        "tabicl_boundary": tab_diag,
        "v2_boundary": v2_diag,
        "blend_boundary": blend_diag,
        "tabicl_residual_corr_vs_v2": residual_corr,
        "tabicl_bootstrap": tab_boot,
        "blend_bootstrap": blend_boot,
        "tabicl_positive_folds": tab_positive_folds,
        "blend_positive_folds": blend_positive_folds,
        "tabicl_worst_fold_gain": tab_worst_fold,
        "blend_worst_fold_gain": blend_worst_fold,
        "tabicl_oof_class_distribution": tab_oof_dist.tolist(),
        "tabicl_test_class_distribution": tab_test_dist.tolist(),
        "tabicl_test_oof_tv": tab_dist_tv,
        "blend_oof_class_distribution": blend_oof_dist.tolist(),
        "blend_test_class_distribution": blend_test_dist.tolist(),
        "blend_test_oof_tv": blend_dist_tv,
        "tabicl_median_thresholds": tab_median_thresholds.tolist(),
        "blend_median_alpha": blend_median_alpha,
        "blend_median_thresholds": blend_median_thresholds.tolist(),
        "tabicl_scientific_positive": tab_scientific_positive,
        "blend_scientific_positive": blend_scientific_positive,
        "tabicl_submission_worthy": tab_submission_worthy,
        "blend_submission_worthy": blend_submission_worthy,
        "breakthrough": breakthrough,
    })

    # Submission creation is gated. No anchor fallback submission exists in V9.
    created_submissions = []
    if tab_submission_worthy:
        sub_tab = pd.DataFrame({CFG.ID_COL: test_uid[CFG.ID_COL].values, CFG.TARGET: tab_test_class.astype(int)})
        assert len(sub_tab) == CFG.EXPECTED_UNIQUE_TEST_IDS and sub_tab[CFG.ID_COL].is_unique
        assert sub_tab[CFG.TARGET].isin([1,2,3,4]).all() and sub_tab.isna().sum().sum() == 0
        sub_tab.to_csv(CFG.OUTPUT_DIR / "submission_v9_tabicl.csv", index=False)
        created_submissions.append(("tabicl", tab_cf, CFG.OUTPUT_DIR / "submission_v9_tabicl.csv"))

    if blend_submission_worthy:
        sub_blend = pd.DataFrame({CFG.ID_COL: test_uid[CFG.ID_COL].values, CFG.TARGET: blend_test_class.astype(int)})
        assert len(sub_blend) == CFG.EXPECTED_UNIQUE_TEST_IDS and sub_blend[CFG.ID_COL].is_unique
        assert sub_blend[CFG.TARGET].isin([1,2,3,4]).all() and sub_blend.isna().sum().sum() == 0
        sub_blend.to_csv(CFG.OUTPUT_DIR / "submission_v9_blend.csv", index=False)
        created_submissions.append(("blend", blend_cf, CFG.OUTPUT_DIR / "submission_v9_blend.csv"))

    if created_submissions:
        RECOMMENDED_CANDIDATE, _, recommended_path = max(created_submissions, key=lambda x: x[1])
        shutil.copyfile(recommended_path, CFG.OUTPUT_DIR / "submission_v9_recommended.csv")
        FULL_SUMMARY["recommended_candidate"] = RECOMMENDED_CANDIDATE
        FULL_SUMMARY["recommended_submission"] = str(CFG.OUTPUT_DIR / "submission_v9_recommended.csv")
    else:
        FULL_SUMMARY["recommended_candidate"] = None
        FULL_SUMMARY["recommended_submission"] = None
        print("Full CV completed, but no candidate passed the preregistered submission gate. No submission CSV was created.")

    display(fold_metrics_df)
    display(blend_selection_df)
    print(json.dumps(FULL_SUMMARY, indent=2))
else:
    FULL_SUMMARY.update({
        "status": "falsified_at_stage_s",
        "recommended_candidate": None,
        "recommended_submission": None,
    })


## 10. Final experiment manifest and artifact audit

The final cell always writes a status manifest. A successful-looking submission cannot appear after a model/runtime failure or Stage-S falsification.


In [24]:
FINAL_MANIFEST = {
    "version": CFG.VERSION,
    "competition": CFG.COMPETITION_NAME,
    "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
    "data_manifest": DATA_MANIFEST,
    "v2_anchor_summary": V2_ANCHOR_SUMMARY,
    "foundation_manifest": FOUNDATION_MANIFEST,
    "stage_s_summary": STAGE_S_SUMMARY,
    "resource_fallback": RESOURCE_FALLBACK,
    "full_summary": FULL_SUMMARY,
    "constraints": [
        "Competition-provided task data only; no external respondent data or test ground truth.",
        "Historical V2 control executes before TabICL installation.",
        "V9 TabICL training/inference rows are one per unique respondent id.",
        "Identifiers id and household_id are excluded from TabICL features.",
        "Fold-selected missing flags and nominal typing use outer-train labels only.",
        "Primary TabICL experiment uses the full feature set; fold-safe top-96 selection is used only if Stage R proves a genuine resource-limit failure.",
        "TabICL full-target regression only; no binary specialist, fine-tuning, pseudo-labeling, or extra model family.",
        "Explicit disk offload directory is provided; runtime/model failure is fatal and never falls back to V2.",
        "Full CV/test inference runs only after preregistered Stage-S promotion.",
        "Headline QWK uses cross-fitted thresholds at unique respondent level.",
        "Blend alpha grid is exactly {0, .25, .50, .75, 1}; alpha and thresholds are selected on the other four folds.",
        "Submission files are created only for candidates passing the preregistered submission gate.",
    ],
}
with open(CFG.OUTPUT_DIR / "v9_summary.json", "w") as f:
    json.dump(FINAL_MANIFEST, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))

files = sorted([p.name for p in CFG.OUTPUT_DIR.iterdir() if p.is_file()])
print("\nV9 artifact files:")
for name in files:
    print(" -", name)

submission_files = [x for x in files if x.startswith("submission_v9") and x.endswith(".csv")]
if not V9_PROMOTED:
    assert not submission_files, "BUG: submission exists despite Stage-S falsification"
elif V9_FULL_AVAILABLE and FULL_SUMMARY.get("recommended_submission") is None:
    assert not submission_files, "BUG: submission exists despite failing submission gate"
else:
    assert "submission_v9_recommended.csv" in submission_files, "Expected recommended submission missing"

print("\nFINAL STATUS")
print("Stage-S promoted:", V9_PROMOTED)
print("Full CV available:", V9_FULL_AVAILABLE)
print("Recommended candidate:", FULL_SUMMARY.get("recommended_candidate"))
print("Recommended submission:", FULL_SUMMARY.get("recommended_submission"))
print("Summary:", CFG.OUTPUT_DIR / "v9_summary.json")



V9 artifact files:
 - data_manifest.json
 - foundation_manifest.json
 - resource_fallback.json
 - runtime_proof.json
 - stage_s_fold0_predictions.npz
 - stage_s_fold2_predictions.npz
 - stage_s_metrics.csv
 - stage_s_runtime_and_preprocessing.json
 - stage_s_summary.json
 - v2_anchor_summary.json
 - v2_anchor_unique_oof.csv
 - v2_anchor_unique_test.csv
 - v9_summary.json

FINAL STATUS
Stage-S promoted: False
Full CV available: False
Recommended candidate: None
Recommended submission: None
Summary: /kaggle/working/oqdq5_v9/v9_summary.json
